# Frame inspector: every validation frame, every model, and why each box was right or wrong

**Inputs (attach both as Kaggle datasets):**
1. **DSERT-RoLL** as you already have it: `processed_data/<Weather>/<drive>/` with `label.pkl`, the LiDAR and radar
   `.npy` files and the camera images. It may be split over several datasets; the notebook finds the pieces.
2. **The four `result.pkl` files** from Snellius. Every evaluation already ran its model on every validation frame and
   saved all predicted boxes there; nothing is run again here. Download them (OnDemand file browser) from
   `~/DSERT-RoLL-3DOD/detection/output/det_model_cfgs/dsert/lidar_only/` and name them after the model:

   | upload as | from |
   |---|---|
   | `radar_scratch.pkl` | `radar_scratch/eval/eval_with_train/epoch_20/val/result.pkl` |
   | `radar_ft_from_livox.pkl` | `radar_ft_from_livox/eval/eval_with_train/epoch_20/val/result.pkl` |
   | `livox_e20_on_radar.pkl` | `livox_e20_on_radar/eval/epoch_20/val/result.pkl` |
   | `livox_pretrain.pkl` | `livox_pretrain/eval/eval_with_train/epoch_20/val/result.pkl` |

   Optional: the `log_train_*.txt` / `log_eval_*.txt` files of those runs, to compare with the official AP.

On a laptop (no `/kaggle/input`) it looks in `~/dsert-roll-sample` and `dsert_results/` instead (see `INPUTS` in section 1).

Run the cells top to bottom. The first run reads every radar and LiDAR file once (section 1) and caches the result in
the output folder; after that everything is fast.

## How the scores work

Same rules as the DSERT-RoLL paper and its evaluation code (official Waymo Open Dataset metrics):
* only labelled and predicted boxes **within 70 m** of the car count (`distance < 70.5 m`, as the code does it)
* a predicted box is **correct** if it overlaps a labelled box of the same class enough: **3D IoU ≥ 0.5 for
  vehicles** (≥ 0.3 for pedestrians and bikes); each labelled box can be taken by **one** predicted box only, and
  boxes are paired by an optimal (Hungarian) assignment within each frame
* only **level-1** labelled boxes are scored; a box that lands on a level-2 one is neither right nor wrong

**The official number (AP, the "0.36" on wandb)** is computed over *all* frames at once, per class: for score
cut-offs 0.00, 0.01, ..., 1.00, keep the boxes above the cut-off, count *precision* = correct / kept boxes and
*recall* = correct / all labelled boxes; **AP = area under that precision-recall curve**. One number for the
whole set; there is no "AP of a frame".

**Per frame (this notebook)** there is **no extra score cut-off**: every box the model output (all boxes in
`result.pkl`, i.e. score ≥ 0.1, the model's own `SCORE_THRESH`) counts, which is the lowest cut-off of the official
sweep. Then for each frame:
* **found** = labelled boxes matched by a predicted box, **missed** = labelled boxes nobody matched,
  **false** = predicted boxes that match no labelled box
* verdict: **GOOD** = everything found and no false boxes, **BAD** = at least one miss,
  **UGLY** = at least one false box (a frame can be BAD+UGLY)

Every miss and every false box gets a **reason** (section 3), and section 4 draws any frame with all of it written
on the picture.

## 1. Load

In [ ]:
import os, io, re, zipfile, pickle, time, hashlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
pd.set_option("display.max_rows", 300); pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

ALL_MODELS = ["radar_scratch", "radar_ft_from_livox", "livox_e20_on_radar", "livox_pretrain"]
NAME = {"radar_scratch": "radar, from scratch", "radar_ft_from_livox": "radar, fine-tuned from LiDAR",
        "livox_e20_on_radar": "LiDAR model fed radar (no training)", "livox_pretrain": "LiDAR model, LiDAR input"}
SENSOR = {m: ("lidar" if m == "livox_pretrain" else "radar") for m in ALL_MODELS}   # what each model sees
CLASSES = ["Vehicle", "Pedestrian", "Bike"]                   # CLASS_NAMES of lidar_only.yaml; id c = CLASSES[c - 1]
IOU_THR = {"Vehicle": 0.5, "Pedestrian": 0.3, "Bike": 0.3}    # same as the official evaluation
MAX_DIST = 70.5                                               # the evaluation ignores everything beyond this
PCR = np.array([0, -76.8, -2, 76.8, 76.8, 4])                 # POINT_CLOUD_RANGE the models were trained with
ON_KAGGLE = os.path.isdir("/kaggle/input")
OUT_DIR = "/kaggle/working" if ON_KAGGLE else "frame_inspector_output"
os.makedirs(OUT_DIR, exist_ok=True)
# Where to look for data and predictions: Kaggle's attached datasets, or on a laptop the DSERT-RoLL sample
# folder and the downloaded result.pkl files (add your own paths here).
INPUTS = ["/kaggle/input"] if ON_KAGGLE else \
    [os.path.expanduser("~/dsert-roll-sample"), "dsert_results", "detection/data/dsert-roll"]

# One walk over the attached datasets: drive folders (and their label.pkl), the result.pkl files, the eval logs.
DRIVE = re.compile(r"\d{4}(_\d{2}){5}$")                      # drive folders are named like 2025_04_16_07_04_30
ROOTS, LABELS, PKLS, LOGS = [], {}, [], []
for path, dirs, files in (w for top in INPUTS if os.path.isdir(top) for w in os.walk(top, followlinks=True)):
    dirs[:] = [d for d in dirs if not d.startswith(".")]     # skip e.g. a download tool's .cache
    for d in [d for d in dirs if DRIVE.match(d)]:
        dirs.remove(d)                                        # do not walk into a drive's thousands of frame files
        root = os.path.dirname(path)                          # .../processed_data (path is .../processed_data/<Weather>)
        if root not in ROOTS:
            ROOTS.append(root)
        if os.path.exists(os.path.join(path, d, "label.pkl")):
            LABELS.setdefault(d, os.path.join(path, d, "label.pkl"))
    PKLS += [os.path.join(path, f) for f in files if f.endswith(".pkl")]
    LOGS += [os.path.join(path, f) for f in files if re.match(r"log_(eval|train)_.*\.txt$", f)]

RESULT = {}
for m in ALL_MODELS:
    found = [p for p in PKLS if os.path.basename(p) == m + ".pkl"] + \
            sorted([p for p in PKLS if os.path.basename(p) == "result.pkl" and m in re.split(r"[\\/]", p)],
                   key=lambda p: "epoch_20" not in p)
    if found:
        RESULT[m] = found[0]
MODELS = list(RESULT)
assert MODELS, "no predictions found: attach the result.pkl files (named radar_scratch.pkl, ...)"
assert LABELS, "no label.pkl found: attach the DSERT-RoLL dataset (processed_data/<Weather>/<drive>/label.pkl)"
for m, p in RESULT.items():
    print("predictions %-20s <- %s" % (m, p))
print("%d drives with labels, data folders: %s" % (len(LABELS), ROOTS))

def resolve(rel):
    "A path from label.pkl (relative to processed_data) -> the file in whichever attached dataset has it ('' if none)."
    for r in ROOTS:
        p = os.path.join(r, rel)
        if os.path.exists(p):
            return p
    return ""

def part(key, k):
    "Rows of a flat array that belong to frame k (k = 0..N-1)."
    off = F[key + "_off"]
    return slice(off[k], off[k + 1])

In [ ]:
# Geometry: 3D IoU of rotated boxes [x, y, z, dx, dy, dz, heading] (z = box centre), as the evaluation computes it
def bev_corners(b):
    "Corners of the box seen from above, counter-clockwise."
    c = np.array([[1, 1], [-1, 1], [-1, -1], [1, -1]]) * [b[3] / 2, b[4] / 2]
    ch, sh = np.cos(b[6]), np.sin(b[6])
    return c @ np.array([[ch, sh], [-sh, ch]]) + b[:2]

def clip_polygon(poly, window):
    "The part of convex polygon `poly` inside convex polygon `window` (both counter-clockwise)."
    for a, b in zip(window, np.roll(window, -1, 0)):
        if not len(poly):
            break
        side = lambda p: (b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0])
        out = []
        for p, q in zip(np.roll(poly, 1, 0), poly):          # edge p -> q
            sp, sq = side(p), side(q)
            if sq >= 0:
                if sp < 0:
                    out.append(p + (q - p) * sp / (sp - sq))
                out.append(q)
            elif sp >= 0:
                out.append(p + (q - p) * sp / (sp - sq))
        poly = np.array(out)
    return poly

def area(poly):
    if len(poly) < 3:
        return 0.0
    x, y = poly[:, 0], poly[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def iou3d(a, b):
    if np.hypot(a[0] - b[0], a[1] - b[1]) > (np.hypot(a[3], a[4]) + np.hypot(b[3], b[4])) / 2:
        return 0.0
    inter = area(clip_polygon(bev_corners(a), bev_corners(b)))
    dz = max(0.0, min(a[2] + a[5] / 2, b[2] + b[5] / 2) - max(a[2] - a[5] / 2, b[2] - b[5] / 2))
    v = inter * dz
    return v / max(a[3] * a[4] * a[5] + b[3] * b[4] * b[5] - v, 1e-9)

def points_in_box(pts, b):
    d = pts[:, :3] - b[:3]
    c, s = np.cos(-b[6]), np.sin(-b[6])
    lx, ly = d[:, 0] * c - d[:, 1] * s, d[:, 0] * s + d[:, 1] * c
    return int(((np.abs(lx) <= b[3] / 2) & (np.abs(ly) <= b[4] / 2) & (np.abs(d[:, 2]) <= b[5] / 2)).sum())

_t = np.array([10, 0, 0, 4, 2, 1.5, 0.])
assert abs(iou3d(_t, _t) - 1) < 1e-9 and abs(iou3d(_t, _t + [1, 0, 0, 0, 0, 0, 0]) - 3 / 5) < 1e-9
assert abs(iou3d(_t, _t + [0, 0, 0, 0, 0, 0, np.pi / 2]) - 1 / 3) < 1e-9   # crossed 4x2 boxes share a 2x2 square

In [ ]:
# Collect every frame the evaluation scored: labelled boxes, radar points, point counts, each model's boxes.
# Same steps as the training / evaluation code (DSERTSingleSensorDataset + dsert_dataset.evaluation):
#   * points: read the .npy, drop empty / NaN points, move radar into LiDAR coordinates through the calibration chain
#     radar -> thermal camera -> LiDAR, keep the point cloud range the models saw
#   * labelled boxes: classes Vehicle / Pedestrian / Bike within 70 m; difficulty 0 = level 1 (scored), else level 2
# The first run reads every radar and LiDAR file once; the result is cached in the output folder.
COUNT_LIDAR = True             # False = do not read the LiDAR files now (faster; LiDAR point counts become -1)
# The cache is named after the drives, prediction files and COUNT_LIDAR it was built from: change any of them
# (e.g. attach more drives) and it is rebuilt instead of silently reusing an old one.
CACHE = os.path.join(OUT_DIR, "frames_cache_%s.npz" % hashlib.md5(repr(
    (sorted(LABELS), sorted((m, os.path.basename(p), os.path.getsize(p)) for m, p in RESULT.items()), COUNT_LIDAR)
).encode()).hexdigest()[:10])
FEATURE_FALLBACKS = ["intensity", "reflectivity", "signal", "doppler"]

def load_points(path, feature):
    "(N, 4) [x, y, z, feature] from a DSERT .npy, as DSERTSingleSensorDataset.load_sensor_points reads it."
    raw = np.load(path)
    if raw.size == 0:
        return np.zeros((0, 4))
    if raw.dtype.names is not None:
        name = next((f for f in [feature] + FEATURE_FALLBACKS if f in raw.dtype.names), "")
        pts = np.stack([raw["x"], raw["y"], raw["z"], raw[name] if name else np.zeros(raw.shape[0])], -1)
    else:
        arr = raw.reshape(-1, raw.shape[-1])
        pts = np.c_[arr[:, :3], arr[:, 3] if arr.shape[1] > 3 else np.zeros(len(arr))]
    pts = pts.astype(np.float64)
    return pts[np.isfinite(pts).all(1) & (np.abs(pts[:, :3]).sum(1) > 0)]

def inverse_T(T):
    T_inv = np.eye(4)
    T_inv[:3, :3] = np.linalg.inv(T[:3, :3])
    T_inv[:3, 3] = -T_inv[:3, :3] @ T[:3, 3]
    return T_inv

def radar_to_lidar(cal):
    "4x4 radar -> Livox transform, as DSERTSingleSensorDataset.sensor_to_livox builds it."
    src = cal.get("Radar", {}) if isinstance(cal.get("Radar"), dict) else {}
    lv = cal.get("Livox", {}) if isinstance(cal.get("Livox"), dict) else {}
    if "Livox" in src:
        return np.asarray(src["Livox"], np.float64)
    if "Radar" in lv:
        return inverse_T(np.asarray(lv["Radar"], np.float64))
    common = [k for k in src if k in lv]
    via = "Thermal_L" if "Thermal_L" in common else common[0]
    return inverse_T(np.asarray(lv[via], np.float64)) @ np.asarray(src[via], np.float64)

def in_range(pts):
    return pts[(pts[:, 0] >= PCR[0]) & (pts[:, 0] < PCR[3]) & (pts[:, 1] >= PCR[1]) & (pts[:, 1] < PCR[4])
               & (pts[:, 2] >= PCR[2]) & (pts[:, 2] < PCR[5])]

def labelled_boxes(info):
    "Boxes the evaluation scores against: (M, 7), class ids (M,), level (M,)."
    a = info.get("annos")
    if not a or len(a["name"]) == 0:
        return np.zeros((0, 7)), np.zeros(0, np.int8), np.zeros(0, np.int8)
    boxes, names = np.asarray(a["gt_boxes_livox"], np.float64), np.asarray(a["name"])
    diff = np.asarray(a.get("difficulty", np.zeros(len(names))), np.int64).reshape(-1)
    diff = np.full(len(names), diff[0]) if diff.size == 1 else diff
    keep = np.isin(names, CLASSES) & (np.hypot(boxes[:, 0], boxes[:, 1]) < MAX_DIST)
    return (boxes[keep, :7], np.array([CLASSES.index(n) + 1 for n in names[keep]], np.int8),
            np.where(np.clip(diff[keep], 0, None) == 0, 1, 2).astype(np.int8))

if os.path.exists(CACHE):
    with np.load(CACHE, allow_pickle=False) as z:
        F = {k: z[k] for k in z.files}
    print("loaded", CACHE, "(delete it to rebuild)")
else:
    preds = {m: {(str(a["sequence_name"]).split("/")[-1], int(a["frame_id"])): a for a in pickle.load(open(p, "rb"))}
             for m, p in RESULT.items()}
    keys = sorted(set().union(*preds.values()))              # (drive, sample) of every frame the evaluation scored
    infos = {}
    for drive in sorted({d for d, _ in keys}):
        if drive in LABELS:
            z = pickle.load(open(LABELS[drive], "rb"))
            for inf in z["info"]:
                inf = dict(inf); inf.update(z["meta"])
                infos[drive, int(inf["sample_idx"])] = inf
    missing = [key for key in keys if key not in infos]
    if missing:
        print("WARNING: %d scored frames have no label.pkl in the attached data (drives: %s); they are skipped"
              % (len(missing), sorted({d for d, _ in missing})))
    keys = [key for key in keys if key in infos]
    assert keys, "none of the predicted frames has a label.pkl in the attached data"

    # Check on the first frame that the files listed in label.pkl are found, before the long pass
    s0 = infos[keys[0]]["sensor"]
    for what, rel in [("radar", s0.get("radar_path", "")), ("LiDAR", s0.get("livox_path", "")),
                      ("camera", s0.get("rgb_left_path", ""))]:
        print("%-6s %s -> %s" % (what, rel, resolve(rel) or "NOT FOUND"))
    assert resolve(s0["radar_path"]), ("radar file not found: label.pkl lists it relative to processed_data, looked in "
                                       "%s. Check that the dataset keeps processed_data/<Weather>/<drive>/..." % ROOTS)
    n_missing = {"radar": 0, "LiDAR": 0, "camera": 0}

    meta = {k: [] for k in ["sample_idx", "sequence", "weather", "light", "has_image", "K", "T_cam", "calib_wh",
                            "img_path", "lidar_path"]}
    flat = {k: [] for k in ["radar_pts", "gt_box", "gt_cls", "gt_level", "gt_radar_pts", "gt_lidar_pts"]}
    for m in MODELS:
        flat.update({"pred_%s_%s" % (m, x): [] for x in ["box", "score", "cls", "pts"]})
    t0 = time.time()
    for n, key in enumerate(keys):
        info = infos[key]
        cal, sensor = info["calibration"], info["sensor"]
        rpath, lpath, ipath = resolve(sensor["radar_path"]), resolve(sensor["livox_path"]), \
            resolve(sensor.get("rgb_left_path", ""))
        for what, p in [("radar", rpath), ("LiDAR", lpath), ("camera", ipath)]:
            n_missing[what] += not p
        rad = load_points(rpath, "doppler") if rpath else np.zeros((0, 4))
        if len(rad):
            rad[:, :3] = rad[:, :3] @ radar_to_lidar(cal)[:3, :3].T + radar_to_lidar(cal)[:3, 3]
        rad = in_range(rad)
        lid = in_range(load_points(lpath, "intensity")) if (COUNT_LIDAR and lpath) else None
        box, cls, level = labelled_boxes(info)
        try:
            K, T_cam = np.asarray(cal["RGB_L"]["intrinsic"], float), np.asarray(cal["Livox"]["RGB_L"], float)
            wh = tuple(cal["RGB_L"].get("shape", (0, 0)))
        except (KeyError, TypeError):
            K, T_cam, wh, ipath = np.zeros((3, 3)), np.zeros((4, 4)), (0, 0), ""
        for k_, v in [("sample_idx", key[1]), ("sequence", str(info.get("sequence_name", key[0]))),
                      ("weather", str(info.get("weather", "?"))), ("light", str(info.get("light", "?"))),
                      ("has_image", bool(ipath)), ("K", K), ("T_cam", T_cam), ("calib_wh", wh),
                      ("img_path", ipath), ("lidar_path", lpath)]:
            meta[k_].append(v)
        flat["radar_pts"].append(rad.astype(np.float32))
        flat["gt_box"].append(box.astype(np.float32)); flat["gt_cls"].append(cls); flat["gt_level"].append(level)
        flat["gt_radar_pts"].append(np.array([points_in_box(rad, b) for b in box], np.int32))
        flat["gt_lidar_pts"].append(np.array([points_in_box(lid, b) if lid is not None else -1 for b in box], np.int32))
        for m in MODELS:
            a = preds[m].get(key)
            ps = np.asarray(a["score"], np.float32).reshape(-1) if a else np.zeros(0, np.float32)
            # boxes_lidar is (n, 7) [x y z dx dy dz heading], or (n, 9) with two velocities: keep the first 7
            pb = np.asarray(a["boxes_lidar"], np.float32).reshape(len(ps), -1)[:, :7] if len(ps) else np.zeros((0, 7), np.float32)
            pc = np.array([CLASSES.index(str(x)) + 1 for x in a["name"]], np.int8) if a else np.zeros(0, np.int8)
            pts = rad if SENSOR[m] == "radar" else lid
            flat["pred_%s_box" % m].append(pb); flat["pred_%s_score" % m].append(ps); flat["pred_%s_cls" % m].append(pc)
            flat["pred_%s_pts" % m].append(np.array([points_in_box(pts, b) if pts is not None else -1 for b in pb],
                                                    np.int32))
        if (n + 1) % 250 == 0 or n + 1 == len(keys):
            print("%d / %d frames, %.0f s" % (n + 1, len(keys), time.time() - t0))

    F = {k: np.array(v) for k, v in meta.items()}
    for k, parts in flat.items():
        F[k] = np.concatenate(parts)
        group = {"radar_pts": "radar", "gt_box": "gt"}.get(k) or (k[:-4] if k.startswith("pred_") and
                                                                   k.endswith("_box") else None)
        if group:
            F[group + "_off"] = np.r_[0, np.cumsum([len(p) for p in parts])]
    F["pred_names"] = np.array(MODELS)
    np.savez(CACHE, **F)
    print("cached in", CACHE)
    if any(n_missing.values()):
        print("WARNING: frames with a file missing in the attached data:", n_missing)

N = len(F["sample_idx"])
MODELS = [str(m) for m in F["pred_names"]]
FR = pd.DataFrame({"drive": [s.split("/")[-1] for s in F["sequence"]], "sample": F["sample_idx"],
                   "weather": F["weather"], "light": F["light"], "has_image": F["has_image"].astype(bool)})
W_ORDER = ["Clear", "Light_Rain", "Heavy_Rain", "Fog", "Light_Snow", "Heavy_Snow"]
WEATHERS = [w for w in W_ORDER if w in set(FR.weather)] + sorted(set(FR.weather) - set(W_ORDER))
print("%d frames (%d with a camera image), %d labelled boxes | models: %s"
      % (N, FR.has_image.sum(), len(F["gt_box"]), MODELS))
display(FR.groupby("weather").agg(frames=("drive", "size"), drives=("drive", "nunique")).reindex(WEATHERS))

In [ ]:
# Match every model's boxes to the labelled boxes, frame by frame (takes a few minutes), the way the official
# evaluation does it (matcher_type TYPE_HUNGARIAN in dsert_eval_detection.py): per frame and class, the one-to-one
# pairing of predicted and labelled boxes with the largest total IoU, using only pairs whose IoU passes the class
# threshold. All boxes the model output take part (no score cut-off; the official lowest cut-off). Labelled boxes of
# difficulty level 2 are matched but not scored at level 1: boxes matching them are 'IGNORED', like in the evaluation.
from scipy.optimize import linear_sum_assignment
GT_ROWS, PRED_ROWS, IOU, STATUS = [], [], {}, {}
for k in range(N):
    gb, gc, gl = F["gt_box"][part("gt", k)], F["gt_cls"][part("gt", k)], F["gt_level"][part("gt", k)]
    gdist = np.hypot(gb[:, 0], gb[:, 1])
    for j in range(len(gb)):
        GT_ROWS.append((k, j, CLASSES[gc[j] - 1], int(gl[j]), gdist[j], int(F["gt_radar_pts"][part("gt", k)][j]),
                        int(F["gt_lidar_pts"][part("gt", k)][j])))
    for m in MODELS:
        s = part("pred_" + m, k)
        pb, ps, pc = F["pred_%s_box" % m][s], F["pred_%s_score" % m][s], F["pred_%s_cls" % m][s]
        inside = np.hypot(pb[:, 0], pb[:, 1]) < MAX_DIST
        I = np.array([[iou3d(p, g) for g in gb] for p in pb]).reshape(len(pb), len(gb))
        IOU[m, k] = I
        status, match = np.where(inside, "FP", "OUT_OF_RANGE").astype(object), np.full(len(pb), -1)
        for c in np.unique(pc):
            pi, gj = np.flatnonzero(inside & (pc == c)), np.flatnonzero(gc == c)
            if not len(pi) or not len(gj):
                continue
            sub = I[np.ix_(pi, gj)]
            ok = sub >= IOU_THR[CLASSES[c - 1]]
            for r, q in zip(*linear_sum_assignment(np.where(ok, sub, 0.0), maximize=True)):
                if ok[r, q]:
                    i, j = pi[r], gj[q]
                    match[i] = j
                    status[i] = "TP" if gl[j] == 1 else "IGNORED"
        STATUS[m, k] = status
        inside_pts = F["pred_%s_pts" % m][s]
        for i in range(len(pb)):
            PRED_ROWS.append((k, m, i, CLASSES[pc[i] - 1], float(ps[i]), status[i], int(match[i]),
                              float(I[i].max()) if len(gb) else 0.0, int(inside_pts[i])))

GT = pd.DataFrame(GT_ROWS, columns=["k", "j", "cls", "level", "dist", "radar_pts", "lidar_pts"])
PRED = pd.DataFrame(PRED_ROWS, columns=["k", "model", "i", "cls", "score", "status", "matched_gt", "best_iou",
                                        "pts_inside"])
GT = GT.merge(FR[["drive", "sample", "weather", "light"]], left_on="k", right_index=True)
PRED = PRED.merge(FR[["drive", "sample", "weather", "light"]], left_on="k", right_index=True)
print("%d labelled boxes (%d level 1), %d predicted boxes" % (len(GT), (GT.level == 1).sum(), len(PRED)))
display(PRED.groupby(["model", "status"]).size().unstack(fill_value=0))

In [ ]:
# What the 70 m cut and the class filter drop: every label in label.pkl for the scored frames, before any filter.
# Only reads the label.pkl files (fast). 'kept' = what the evaluation scores (and this notebook counts above).
raw = []
scored = set(zip(FR.drive, FR["sample"]))
for drive in sorted(set(FR.drive)):
    z = pickle.load(open(LABELS[drive], "rb"))
    for inf in z["info"]:
        if (drive, int(inf["sample_idx"])) not in scored or not inf.get("annos") or not len(inf["annos"]["name"]):
            continue
        a = inf["annos"]
        b = np.asarray(a["gt_boxes_livox"], np.float64)
        d = np.hypot(b[:, 0], b[:, 1])
        diff = np.asarray(a.get("difficulty", np.zeros(len(d))), np.int64).reshape(-1)
        diff = np.full(len(d), diff[0]) if diff.size == 1 else diff
        for n, dd, x, df in zip(a["name"], d, b[:, 0], diff):
            raw.append((drive, str(n), dd, x, int(df)))
RAW = pd.DataFrame(raw, columns=["drive", "name", "dist", "x", "difficulty"]).merge(
    FR.drop_duplicates("drive")[["drive", "weather"]], on="drive")
RAW["fate"] = np.select([~RAW.name.isin(CLASSES), RAW.dist >= MAX_DIST, RAW.x < PCR[0]],
                         ["dropped: class not scored", "dropped: beyond 70 m", "kept, but behind the car (x < 0)"],
                         "kept")
print("%d labels in label.pkl for the %d scored frames; %d kept (scored), %d dropped."
      % (len(RAW), N, RAW.fate.str.startswith("kept").sum(), RAW.fate.str.startswith("dropped").sum()))
print("'behind the car': scored, but outside the models' point range (x from %g m), so no model can find them."
      % PCR[0])
display(RAW.groupby(["name", "fate"]).size().unstack(fill_value=0))
print("Per weather:")
display(RAW.groupby(["weather", "fate"]).size().unstack(fill_value=0).reindex(WEATHERS))
print("Distance of all labels (m):")
display(pd.cut(RAW.dist, [0, 20, 40, 60, 70.5, 80, 100, 150, np.inf], right=False).value_counts(sort=False))
print("difficulty values in label.pkl:", RAW.difficulty.value_counts().to_dict())

## 2. AP over all frames, and per weather

In [ ]:
from scipy.optimize import linear_sum_assignment
# AP the way the official evaluation computes it (Waymo Open Dataset detection metrics, dsert_eval_detection.py):
#  * score cut-offs 0.00, 0.01, ..., 0.99, 1.00; at each one keep the predicted boxes with score >= cut-off and
#    redo the Hungarian matching per frame and class (so a high-score box can be the hit at a high cut-off even if a
#    better-overlapping low-score box takes the object at cut-off 0)
#  * precision = TP / (TP + FP), recall = TP / labelled boxes, at each cut-off (precision 1 where recall is 0)
#  * AP = Waymo's ComputeMeanAveragePrecision: precision made non-increasing, recall gaps filled every 0.05,
#    trapezoid area
CUTOFFS = np.r_[np.arange(100) * 0.01, 1.0].astype(np.float32)

def tp_fp_per_cutoff(m, cls):
    "(frames, cut-offs) arrays of TP and FP counts for model m and class cls, matching redone at every cut-off."
    c = CLASSES.index(cls) + 1
    TP = np.zeros((N, len(CUTOFFS)), np.int32); FP = np.zeros((N, len(CUTOFFS)), np.int32)
    for k in range(N):
        s = part("pred_" + m, k)
        pb, ps, pc = F["pred_%s_box" % m][s], F["pred_%s_score" % m][s], F["pred_%s_cls" % m][s]
        keep = (pc == c) & (np.hypot(pb[:, 0], pb[:, 1]) < MAX_DIST)
        if not keep.any():
            continue
        pi = np.flatnonzero(keep)
        gc, gl = F["gt_cls"][part("gt", k)], F["gt_level"][part("gt", k)]
        gj = np.flatnonzero(gc == c)
        above = ps[pi][None, :] >= CUTOFFS[:, None]                       # (cut-offs, kept boxes)
        n_above = above.sum(1)
        tp = np.zeros(len(CUTOFFS), np.int32); ign = np.zeros(len(CUTOFFS), np.int32)
        if len(gj):
            sub = IOU[m, k][np.ix_(pi, gj)]
            ok = sub >= IOU_THR[cls]
            cand = np.flatnonzero(ok.any(1))                                # boxes that can match anything
            if len(cand):
                cand = cand[np.argsort(-ps[pi][cand], kind="stable")]       # by score, highest first
                n_cand = above[:, cand].sum(1)                              # kept candidates at each cut-off
                for n in np.unique(n_cand):                                 # = the top n by score
                    r_, q_ = linear_sum_assignment(np.where(ok[cand[:n]], sub[cand[:n]], 0.0), maximize=True)
                    hit = [q for r, q in zip(r_, q_) if ok[cand[r], q]]
                    tp[n_cand == n] = sum(gl[gj[q]] == 1 for q in hit)
                    ign[n_cand == n] = sum(gl[gj[q]] != 1 for q in hit)
        TP[k], FP[k] = tp, n_above - tp - ign
    return TP, FP

def waymo_ap(precision, recall, delta=0.05):
    "Waymo's ComputeMeanAveragePrecision (metrics_utils.cc), line by line."
    rp = {0.0: 1.0}
    for p, r in zip(precision, recall):
        rp[float(r)] = max(rp.get(float(r), 0.0), float(p))
    pr, last, best = [], min(rp), 0.0
    for r in sorted(rp, reverse=True):
        while last - r > delta + 1e-6:
            last -= delta
            pr.append((best, last))
        best = max(rp[r], best)
        pr.append((best, r))
        last = r
    if len(pr) >= 2:
        pr[-1] = (pr[-2][0], pr[-1][1])
    return sum(0.5 * (pr[i - 1][1] - pr[i][1]) * (pr[i - 1][0] + pr[i][0]) for i in range(1, len(pr)))

def ap_of(TP, FP, n_gt):
    tp, fp = TP.sum(0).astype(np.float32), FP.sum(0).astype(np.float32)
    recall = tp / n_gt if n_gt else np.zeros_like(tp)
    precision = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0).astype(np.float32)
    precision = np.where(recall == 0, 1.0, precision)
    return waymo_ap(precision, recall), recall[0], (tp[0] / (tp[0] + fp[0]) if tp[0] + fp[0] else np.nan)

GROUPS = {"all": np.ones(N, bool)} | {w: (FR.weather == w).values for w in WEATHERS}
rows, AP = [], {}
for m in MODELS:
    for cls in CLASSES:
        TP, FP = tp_fp_per_cutoff(m, cls)
        n_gt_frame = GT[(GT.cls == cls) & (GT.level == 1)].groupby("k").size().reindex(range(N), fill_value=0).values
        row = {"model": m, "class": cls}
        for g, mask in GROUPS.items():
            AP[m, cls, g], rec, prec = ap_of(TP[mask], FP[mask], n_gt_frame[mask].sum())
            row["AP " + g] = round(AP[m, cls, g], 4)
            if g == "all":
                row.update({"labelled": int(n_gt_frame.sum()), "predicted": int(TP[:, 0].sum() + FP[:, 0].sum()),
                            "recall (all boxes)": round(float(rec), 3), "precision (all boxes)": round(float(prec), 3)})
        rows.append(row)
print("AP per class, over all frames and per weather (frames per weather: %s)"
      % ", ".join("%s %d" % (w, GROUPS[w].sum()) for w in WEATHERS))
T = pd.DataFrame(rows).set_index(["model", "class"])
display(T[[c for c in T if c.startswith("AP ")] + [c for c in T if not c.startswith("AP ")]])

if {"radar_scratch", "radar_ft_from_livox"} <= set(MODELS):
    v = pd.DataFrame({NAME.get(m, m): {g: AP[m, "Vehicle", g] for g in GROUPS} for m in MODELS}).T
    v.loc["fine-tuned minus scratch"] = v.loc[NAME["radar_ft_from_livox"]] - v.loc[NAME["radar_scratch"]]
    print("\nVehicle AP per weather, and what the LiDAR pre-training adds to the radar model:")
    display(v.round(4))

# No score cut-off on the per-frame pages: every box in result.pkl counts (the official lowest cut-off).
# Raising a value here by hand hides boxes below it (drawn grey dashed); the matching above stays the one with all boxes.
CUT = {m: 0.0 for m in MODELS}

## 3. Why each box was missed or false

**Missed labelled box**: the first reason in this list that applies.

| reason | meaning |
|---|---|
| `no <sensor> points on it` | not a single radar (or LiDAR) point hit the object: the model had nothing to see |
| `box shifted / rotated / wrong size` | a box sits on it but does not overlap enough; the biggest error is named |
| `box went to a neighbour` | the nearby box was used for another labelled object |
| `seen as another class` | a box of the wrong class sits on it |
| `nothing predicted near it` | the model saw nothing there (points present, but no box within 4 m) |

`right box, score too low` and `only a weak guess nearby` can only appear if you raise `CUT` by hand in section 2.
With the paper's rule (no cut-off, every box counts) they never appear.

**False box** (a predicted box that is not kept):

| reason | meaning |
|---|---|
| `duplicate` | a second box on an object another box already took |
| `misplaced box on a real object` | it overlaps a labelled object a little (IoU 0.1 or more), but not enough |
| `ghost on clutter` | sensor points inside, but no labelled object; check the camera: it may be an unlabelled car, a pole, a wall... |
| `ghost in empty space` | no sensor points inside at all |

In [ ]:
def miss_reason(m, g, cut):
    "Why model m missed labelled box g (a row of GT) at score cut-off `cut`: (reason, IoU of the closest box)."
    k, j = g.k, g.j
    gb = F["gt_box"][part("gt", k)][j]
    s = part("pred_" + m, k)
    pb, ps, pc = F["pred_%s_box" % m][s], F["pred_%s_score" % m][s], F["pred_%s_cls" % m][s]
    I = IOU[m, k][:, j] if len(pb) else np.zeros(0)
    same = pc == CLASSES.index(g.cls) + 1
    near = np.hypot(pb[:, 0] - gb[0], pb[:, 1] - gb[1]) < 4 if len(pb) else np.zeros(0, bool)
    pts = g.radar_pts if SENSOR[m] == "radar" else g.lidar_pts
    if (same & (I >= IOU_THR[g.cls]) & (ps < cut)).any():
        return "right box, score too low", float(I[same & (ps < cut)].max())
    if pts == 0:
        return "no %s points on it" % SENSOR[m], np.nan
    cand = np.flatnonzero(same & near & (ps >= cut))
    if len(cand):
        i = cand[np.argmax(I[cand])]
        if STATUS[m, k][i] in ("TP", "IGNORED") and I[i] < IOU_THR[g.cls]:
            return "box went to a neighbour", float(I[i])
        shift = np.hypot(*(pb[i][:2] - gb[:2]))
        turn = np.degrees(abs((pb[i][6] - gb[6] + np.pi / 2) % np.pi - np.pi / 2))
        size = max(abs(pb[i][3] - gb[3]) / gb[3], abs(pb[i][4] - gb[4]) / gb[4])
        worst = max([(shift / 1.0, "box shifted %.1f m" % shift), (turn / 20, "box rotated %.0f deg" % turn),
                     (size / 0.25, "box wrong size (%.0f%%)" % (100 * size))])
        return worst[1], float(I[i])
    if ((~same) & (I >= 0.3) & (ps >= cut)).any():
        return "seen as another class", np.nan
    if (near & (ps < cut)).any():
        return "only a weak guess nearby", float(I[near].max())
    return "nothing predicted near it", float(I.max()) if len(I) else 0.0

def false_reason(row):
    "Why a false box (row of PRED with status FP) is false."
    I = IOU[row.model, row.k][row.i]
    same = F["gt_cls"][part("gt", row.k)] == CLASSES.index(row.cls) + 1
    if (same & (I >= IOU_THR[row.cls])).any():      # right place, but that object was already taken
        return "duplicate"
    if row.best_iou >= 0.1:
        return "misplaced box on a real object"
    return "ghost on clutter" if row.pts_inside > 0 else "ghost in empty space"

def explain(cut=None):
    cut = cut or CUT
    miss = []
    for m in MODELS:
        p = PRED[(PRED.model == m) & (PRED.score >= cut[m]) & PRED.status.isin(["TP", "IGNORED"])]
        found = set(zip(p.k, p.matched_gt))
        for g in GT[GT.level == 1].itertuples():
            if (g.k, g.j) not in found:
                r, iou = miss_reason(m, g, cut[m])
                miss.append((m, g.k, g.j, r, re.sub(r" [0-9.]+ ?(m|deg)$| \([0-9]+%\)$", "", r), iou))
    MISS = pd.DataFrame(miss, columns=["model", "k", "j", "reason", "reason_group", "iou"]).merge(
        GT, on=["k", "j"])
    f = PRED[PRED.status.eq("FP") & (PRED.score >= PRED.model.map(cut))].copy()
    f["reason_group"] = [false_reason(r) for r in f.itertuples()]
    return MISS, f

MISS, FALSE = explain()
order_m = ["no radar points on it", "no lidar points on it", "nothing predicted near it", "only a weak guess nearby",
           "right box, score too low", "box shifted", "box rotated", "box wrong size", "box went to a neighbour",
           "seen as another class"]
order_f = ["ghost in empty space", "ghost on clutter", "misplaced box on a real object", "duplicate"]
tab_m = MISS.groupby(["model", "reason_group"]).size().unstack(fill_value=0)
tab_f = FALSE.groupby(["model", "reason_group"]).size().unstack(fill_value=0)
print("MISSED labelled boxes, by reason (all classes, level 1):")
display(tab_m.reindex(columns=[c for c in order_m if c in tab_m.columns]))
print("FALSE boxes, by reason:")
display(tab_f.reindex(columns=[c for c in order_f if c in tab_f.columns]))

REASON_COLOR = dict(zip(order_m + order_f, list(plt.cm.tab10.colors) + list(plt.cm.Set2.colors)))
fig, axes = plt.subplots(1, 2, figsize=(22, 0.9 * len(MODELS) + 3))
for ax, tab, order, title in [(axes[0], tab_m, order_m, "missed labelled boxes"), (axes[1], tab_f, order_f, "false boxes")]:
    cols = [c for c in order if c in tab.columns]
    t = tab.reindex(index=MODELS, columns=cols).fillna(0)
    left = np.zeros(len(t))
    for c in cols:
        ax.barh([NAME.get(m, m) for m in t.index], t[c], left=left, color=REASON_COLOR[c], label=c)
        left += t[c].values
    ax.set_title(title + " per model, by reason (number of boxes)", fontsize=13)
    ax.legend(fontsize=10, bbox_to_anchor=(0, -0.1), loc="upper left", ncol=3)
    ax.invert_yaxis()
plt.tight_layout(); plt.show()

print("The same per weather (missed boxes, share of each reason):")
display((MISS.groupby(["model", "weather", "reason_group"]).size().unstack(fill_value=0)
         .pipe(lambda t: t.div(t.sum(1), axis=0) * 100).round(0)))

## 3b. Recall by angle: do the radar models lose objects at the sides?

**Angle** = direction of a labelled box's centre seen from the Livox: 0° = straight ahead, + = left, − = right.
The sensors see different widths (DSERT-RoLL sensor table): the **radar** (RETINA-4FN) about **±50°**, the **Livox**
about **±60°**, while the models' point range lets boxes reach ±90°. If the radar's narrower view costs detections,
then beyond ~50°:
* the share of labelled boxes with **no radar point** on them goes up, and
* the **recall** (share of labelled boxes found, counting every box the model output) of the radar models drops
  while the LiDAR model's holds up to ~60°.

Wide-angle objects are mostly close by, and close objects are easier, so the second table splits by distance too.
The last tables say *why* the radar models miss at each angle (section 3's reasons) and where their false boxes sit.
Small bins (few boxes) are noisy: the `boxes` column says how many there are.

In [ ]:
ANGLE_BINS = [0, 10, 20, 30, 40, 50, 60, 90]        # |angle| in degrees
ANGLE_CLASSES = CLASSES                              # e.g. ["Vehicle"] for cars only
DIST_BINS = [0, 20, 40, 70.5]
RADAR_HALF_FOV, LIDAR_HALF_FOV = 50, 60              # RETINA-4FN 100 deg wide, Livox HAP 120 deg wide
A_MODELS = [m for m in ["radar_scratch", "radar_ft_from_livox", "livox_e20_on_radar", "livox_pretrain"] if m in MODELS]
bin_names = ["%d-%d°" % (a, b) for a, b in zip(ANGLE_BINS[:-1], ANGLE_BINS[1:])]

def box_angle(boxes):
    return np.degrees(np.arctan2(boxes[:, 1], boxes[:, 0]))

# Angle and distance of every labelled box (GT row k, j = box j of frame k)
gb_all = F["gt_box"][F["gt_off"][GT.k.values] + GT.j.values]
GT["angle"] = box_angle(gb_all)
G = GT[(GT.level == 1) & GT.cls.isin(ANGLE_CLASSES)].copy()
G["angle_bin"] = pd.cut(G.angle.abs(), ANGLE_BINS, right=False, labels=bin_names)
G["dist_bin"] = pd.cut(G.dist, DIST_BINS, right=False, labels=["%d-%dm" % (a, b) for a, b in zip(DIST_BINS[:-1], [20, 40, 70])])

def found_at_cut(m, cut=None):
    "True for each row of G that model m found (with any box, or above CUT if you raised it in section 2)."
    cut = cut or CUT
    p = PRED[(PRED.model == m) & (PRED.score >= cut[m]) & PRED.status.isin(["TP", "IGNORED"])]
    found = set(zip(p.k, p.matched_gt))
    return np.array([(k, j) in found for k, j in zip(G.k, G.j)])

for m in A_MODELS:
    G["found_" + m] = found_at_cut(m)

# 1. Recall per angle bin, next to how many boxes have any radar / LiDAR point on them
t = G.groupby("angle_bin", observed=False).agg(boxes=("k", "size"),
                                               with_radar_pts=("radar_pts", lambda x: (x > 0).mean()))
if (G.lidar_pts >= 0).any():
    t["with_lidar_pts"] = G.groupby("angle_bin", observed=False).lidar_pts.apply(lambda x: (x > 0).mean())
for m in A_MODELS:
    t["recall " + m] = G.groupby("angle_bin", observed=False)["found_" + m].mean()
if {"radar_scratch", "radar_ft_from_livox"} <= set(A_MODELS):
    t["ft - scratch"] = t["recall radar_ft_from_livox"] - t["recall radar_scratch"]
print("Labelled boxes (%s, level 1) by |angle|: share with points on them, and recall"
      % ", ".join(ANGLE_CLASSES))
display(t.style.format({c: "{:.0%}" for c in t.columns if c != "boxes"} | {"ft - scratch": "{:+.0%}"}))

fig, ax = plt.subplots(figsize=(10, 5.5))
mid = (np.array(ANGLE_BINS[:-1]) + np.array(ANGLE_BINS[1:])) / 2
for m in A_MODELS:
    ax.plot(mid, 100 * t["recall " + m].values, "-o", lw=2, label="recall: " + NAME.get(m, m))
ax.plot(mid, 100 * t.with_radar_pts.values, "k--", lw=1.5, label="boxes with >= 1 radar point")
if "with_lidar_pts" in t:
    ax.plot(mid, 100 * t.with_lidar_pts.values, ":", c="0.4", lw=1.5, label="boxes with >= 1 LiDAR point")
for x, name in [(RADAR_HALF_FOV, "radar edge"), (LIDAR_HALF_FOV, "Livox edge")]:
    ax.axvline(x, c="0.6", lw=1)
    ax.text(x + 0.5, 101, name, fontsize=9, color="0.4", va="bottom")
for x, n in zip(mid, t.boxes.values):
    ax.text(x, -9, "n=%d" % n, ha="center", fontsize=8, color="0.4")
ax.set_xlabel("|angle| of the labelled box from straight ahead (degrees)"); ax.set_ylabel("%")
ax.set_ylim(-12, 108); ax.set_xlim(0, 90); ax.grid(alpha=.3)
ax.legend(fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
ax.set_title("Recall by angle (%s)" % ", ".join(ANGLE_CLASSES)); plt.show()

# 2. The same split by distance (wide angles are mostly close, which is easier)
print("Recall by |angle| and distance, 'recall (boxes)':")
tabs = {}
for m in A_MODELS:
    g = G.groupby(["dist_bin", "angle_bin"], observed=False)["found_" + m].agg(["mean", "size"])
    cell = g.apply(lambda r: "%.0f%% (%d)" % (100 * r["mean"], r["size"]) if r["size"] else "-", axis=1)
    tabs[NAME.get(m, m)] = cell.unstack("angle_bin").reindex(columns=bin_names)
display(pd.concat(tabs, names=["model", "distance"]))

# 3. Why the radar models miss, per angle bin (share of each section-3 reason)
M = MISS.merge(G[["k", "j", "angle_bin"]], on=["k", "j"])
for m in [m for m in A_MODELS if SENSOR[m] == "radar"]:
    tab = M[M.model == m].groupby(["angle_bin", "reason_group"], observed=False).size().unstack(fill_value=0)
    tab = tab.reindex(columns=[c for c in order_m if c in tab.columns])
    print("\n%s: why labelled boxes were missed, per |angle| (share of the misses in that bin, number of misses)"
          % NAME.get(m, m))
    display(tab.div(tab.sum(1).replace(0, np.nan), axis=0).mul(100).round(0).assign(misses=tab.sum(1)))

# 4. Where the false boxes are: per angle bin, false boxes per 100 frames (angle of the predicted box)
fa = []
for r in FALSE.itertuples():
    b = F["pred_%s_box" % r.model][F["pred_%s_off" % r.model][r.k] + r.i]
    fa.append(abs(box_angle(b[None])[0]))
FALSE["angle_bin"] = pd.cut(fa, ANGLE_BINS, right=False, labels=bin_names) if len(fa) else []
print("\nFalse boxes per 100 frames, by |angle| of the false box:")
display((FALSE.groupby(["model", "angle_bin"], observed=False).size().unstack(fill_value=0) * 100 / N).round(1)
        .reindex(A_MODELS))

# 5. Frames to look at: wide-angle boxes the LiDAR model found but the fine-tuned radar model missed
if {"radar_ft_from_livox", "livox_pretrain"} <= set(A_MODELS):
    WIDE = G[(G.angle.abs() >= RADAR_HALF_FOV) & G.found_livox_pretrain & ~G.found_radar_ft_from_livox]
    print("\n%d wide-angle (>= %d deg) boxes found by the LiDAR model but missed by the fine-tuned radar model."
          % (len(WIDE), RADAR_HALF_FOV), "Draw one after running section 4 with show(k), e.g. k in",
          [int(k) for k in sorted(WIDE.k.unique())[:10]])

## 3c. Labelled boxes the radar cannot see at all

Boxes with **not one radar point** on them (the radar points the models saw: moved into LiDAR coordinates, inside the
point range). The radar models have nothing to go on for these, so they cap radar recall. The cell prints:
1. **how many**: boxes, frames they are in, frames where no labelled box has a radar point, frames where the radar
   returned nothing at all
2. **found anyway**: per model, how many of these boxes it still matched (IoU and score of the hit), next to its
   recall on boxes that do have radar points. The LiDAR model is the reference: the object is there, the radar just
   did not see it
3. **per weather**: frames, boxes, zero-point boxes, and found-anyway per model, all as numbers
4. **what kind of boxes**: class, distance, angle, LiDAR points on the box (0 LiDAR points too = probably hidden or a
   doubtful label)
5. **drives** with the most of them
6. **frames to look at** with `show(k)`, including every zero-point box a radar model still found

In [ ]:
# Labelled boxes with NO radar point on them (after moving radar into LiDAR coordinates and the point-range cut,
# exactly the points the radar models saw). For these the radar models have nothing to see.
# Needs sections 1 (F, FR, GT) and 3 (MISS) to have run.
L1 = GT[GT.level == 1].copy()
gb = F["gt_box"][F["gt_off"][L1.k.values] + L1.j.values]
L1["angle"] = np.degrees(np.arctan2(gb[:, 1], gb[:, 0]))
L1["zero"] = L1.radar_pts == 0
n_radar = pd.Series(np.diff(F["radar_off"]), name="radar_pts_in_frame")      # radar points per frame (all of them)
fr = L1.groupby("k").agg(boxes=("j", "size"), zero=("zero", "sum")).join(n_radar).join(FR)
fr["all_zero"] = fr.zero == fr.boxes                                          # not one labelled box has a radar point
fr["radar_empty"] = fr.radar_pts_in_frame == 0                                # the radar gave no points at all
Z = L1[L1.zero].copy()

# Which model found which box (matched by one of its boxes, every box counted), with that box's IoU and score
HIT = {}
for m in MODELS:
    p = PRED[(PRED.model == m) & (PRED.score >= CUT[m]) & PRED.status.isin(["TP", "IGNORED"])]
    HIT[m] = {(r.k, r.matched_gt): (float(IOU[m, r.k][r.i, r.matched_gt]), r.score) for r in p.itertuples()}
    L1["found_" + m] = [(k, j) in HIT[m] for k, j in zip(L1.k, L1.j)]
    Z["found_" + m] = L1.loc[Z.index, "found_" + m]

# Radar points close to each zero-point box (centre within 5 m): what a radar model could have guessed from
def radar_near(k, j, r=5.0):
    b = F["gt_box"][part("gt", k)][j]
    p = F["radar_pts"][part("radar", k)]
    return int((np.hypot(p[:, 0] - b[0], p[:, 1] - b[1]) < r).sum())
Z["radar_within_5m"] = [radar_near(k, j) for k, j in zip(Z.k, Z.j)]

print("1. HOW MANY")
print("   %d of %d labelled boxes (%.1f%%) have no radar point on them."
      % (len(Z), len(L1), 100 * len(Z) / len(L1)))
print("   They are in %d of the %d frames with labelled boxes (%.1f%%): %s."
      % ((fr.zero > 0).sum(), len(fr), 100 * (fr.zero > 0).mean(),
         ", ".join("%d frames with %d such box%s" % (v, n, "" if n == 1 else "es")
                   for n, v in fr.loc[fr.zero > 0, "zero"].value_counts().sort_index().items())))
print("   In %d frames NOT ONE labelled box has a radar point. In %d frames the radar returned no point at all "
      "(%d zero-point boxes are in those)." % (fr.all_zero.sum(), fr.radar_empty.sum(), fr.loc[fr.radar_empty, "zero"].sum()))

print("\n2. FOUND ANYWAY: labelled boxes with no radar point that a model still matched (every box counted)")
rows = []
for m in MODELS:
    f = Z[Z["found_" + m]]
    hits = [HIT[m][k, j] for k, j in zip(f.k, f.j)]
    with_pts = L1[~L1.zero]
    rows.append({"model": NAME.get(m, m), "sees": SENSOR[m],
                 "found (of %d zero-point boxes)" % len(Z): len(f),
                 "recall on zero-point boxes": "%.1f%%" % (100 * len(f) / max(len(Z), 1)),
                 "recall on boxes WITH radar points": "%.1f%%" % (100 * with_pts["found_" + m].mean()),
                 "median IoU of the hit": round(float(np.median([h[0] for h in hits])), 2) if hits else np.nan,
                 "median score of the hit": round(float(np.median([h[1] for h in hits])), 2) if hits else np.nan})
display(pd.DataFrame(rows).set_index("model"))
print("   Radar points within 5 m of the zero-point boxes: found by a radar model %s, missed by both radar models %s"
      % ("median %.0f" % Z[Z.found_radar_scratch | Z.found_radar_ft_from_livox].radar_within_5m.median()
         if (Z.found_radar_scratch | Z.found_radar_ft_from_livox).any() else "none found",
         "median %.0f" % Z[~Z.found_radar_scratch & ~Z.found_radar_ft_from_livox].radar_within_5m.median()))

print("\n3. PER WEATHER (numbers, then found-anyway per model)")
w = fr.groupby("weather").agg(frames=("boxes", "size"), frames_with_zero=("zero", lambda x: (x > 0).sum()),
                              frames_all_zero=("all_zero", "sum"), frames_radar_empty=("radar_empty", "sum"),
                              boxes=("boxes", "sum"), zero_boxes=("zero", "sum"),
                              median_radar_pts_per_frame=("radar_pts_in_frame", "median"))
w["% of boxes zero"] = (100 * w.zero_boxes / w.boxes).round(1)
w["% of all zero boxes"] = (100 * w.zero_boxes / max(len(Z), 1)).round(1)
for m in MODELS:
    w["found: " + m] = Z.groupby("weather")["found_" + m].sum().reindex(w.index).fillna(0).astype(int)
display(w.reindex([x for x in WEATHERS if x in w.index]))

print("\n4. WHAT KIND OF BOXES (how many have no radar point, and how many each model still found)")
for name, col in [("class", L1.cls),
                  ("distance", pd.cut(L1.dist, [0, 10, 20, 40, 70.5], right=False)),
                  ("|angle| (radar sees about +-50 deg)", pd.cut(L1.angle.abs(), [0, 30, 50, 60, 90], right=False)),
                  ("LiDAR points on the box", pd.cut(L1.lidar_pts, [-1, 0, 10, 100, 1e9], right=True,
                                                     labels=["0", "1-10", "11-100", ">100"]))]:
    g = L1.groupby(col, observed=False)
    t = g.zero.agg(boxes="size", zero_boxes="sum")
    t["% zero"] = (100 * t.zero_boxes / t.boxes.replace(0, np.nan)).round(1)
    for m in MODELS:
        t["found: " + m] = L1[L1.zero].groupby(col[L1.zero], observed=False)["found_" + m].sum().reindex(t.index).fillna(0).astype(int)
    print("  by", name); display(t)

print("\n5. DRIVES with the most zero-point boxes")
d = fr.groupby(["drive", "weather"]).agg(frames=("boxes", "size"), frames_with_zero=("zero", lambda x: (x > 0).sum()),
                                         boxes=("boxes", "sum"), zero_boxes=("zero", "sum"),
                                         radar_empty_frames=("radar_empty", "sum"))
d["% of boxes zero"] = (100 * d.zero_boxes / d.boxes).round(1)
display(d.sort_values("zero_boxes", ascending=False).head(12))

print("\n6. FRAMES TO LOOK AT with show(k)")
ZERO_FRAMES = fr[fr.zero > 0].sort_values("zero", ascending=False)
print("   most zero-point boxes:", [int(k) for k in ZERO_FRAMES.index[:10]])
for m in [m for m in MODELS if SENSOR[m] == "radar"]:
    f = Z[Z["found_" + m]]
    print("   zero-point boxes %s still found:" % NAME.get(m, m))
    if len(f):
        display(f.assign(iou=[HIT[m][k, j][0] for k, j in zip(f.k, f.j)],
                         score=[HIT[m][k, j][1] for k, j in zip(f.k, f.j)])
                [["k", "j", "drive", "sample", "weather", "cls", "dist", "angle", "lidar_pts", "radar_within_5m",
                  "iou", "score"]].round(2).sort_values("score", ascending=False).head(15))
    else:
        print("   none")

## 4. Look at frames

`show(frame)` draws one frame for the given models side by side:
* top row, three pictures: the **camera image as it is**; the same with **only the LiDAR points** (small grey dots);
  the same with **only the radar points** (big dots, coloured by distance from the car: blue near, green/yellow
  middle, orange/red far, up to 70 m; colour bar underneath)
* per model: the camera image with the points of the sensor that model sees (radar for the radar models, LiDAR for
  the LiDAR model; `both_sensors=True` draws both) and all its boxes in 3D; below it the **top view** (both sensors);
  below that the **list** of every labelled object and every predicted box

Names: labelled objects are **L1, L2, ...**; the model's boxes are **a, b, c, ...** by score (a = highest).
Colours: labelled object **green** = found, **red** = missed; predicted box **blue** = **kept** (paired with a labelled
object, counts as correct, drawn as `c=L1`), **magenta** = **not kept** (a false box; the list says why: duplicate,
misplaced box, ghost on clutter, ghost in empty space).

"Kept" is the pairing with every box counted (score cut-off 0). The official AP redoes the pairing at every score
cut-off, so a stronger box can take the object at higher cut-offs: the line `paired at each cut-off` under each object
shows which box holds it from 0.00 to 1.00, e.g. `c 0.00-0.37, a 0.38-0.56, nobody 0.57-1.00`.

In [ ]:
from matplotlib.lines import Line2D
from IPython.display import Image as ShowImage

# Every picture show() draws is saved as a .jpg under pictures/<folder>/ in the output folder (Kaggle: /kaggle/working,
# downloadable from the Output tab; zip_pictures() at the end puts them all in one pictures.zip).
PIC_DIR = os.path.join(OUT_DIR, "pictures")
SAVE_DPI = 80          # resolution of the saved pictures (80 -> about 2000 pixels wide for two models)
SHOW_WIDTH = 1400      # width of the copy shown in the notebook (pixels); the saved .jpg keeps SAVE_DPI

def zip_pictures():
    "All saved pictures -> pictures.zip in the output folder."
    out = os.path.join(OUT_DIR, "pictures.zip")
    files = [os.path.join(r, f) for r, _, fs in os.walk(PIC_DIR) for f in fs] if os.path.isdir(PIC_DIR) else []
    with zipfile.ZipFile(out, "w") as zf:
        for f in sorted(files):
            zf.write(f, os.path.relpath(f, PIC_DIR))
    print("%d pictures in %s" % (len(files), out))

# Per frame and model: labelled / found / false / missed, every box counted (the official lowest cut-off).
# Used for the picture titles and to pick frames in sections 5 and 6.
def frame_scores(cut=None):
    cut = cut or CUT
    rows = []
    for m in MODELS:
        p = PRED[(PRED.model == m) & (PRED.score >= cut[m])]
        tp = p[p.status == "TP"].groupby("k").size()
        fp = p[p.status == "FP"].groupby("k").size()
        n = GT[GT.level == 1].groupby("k").size()
        d = pd.DataFrame({"labelled": n, "found": tp, "false": fp}).reindex(range(N)).fillna(0).astype(int)
        d["missed"] = d.labelled - d.found
        d["verdict"] = np.select([(d.missed > 0) & (d.false > 0), d.missed > 0, d.false > 0],
                                 ["BAD+UGLY", "BAD", "UGLY"], "GOOD")
        d["model"] = m
        rows.append(d.reset_index().rename(columns={"index": "k"}))
    return pd.concat(rows, ignore_index=True).merge(FR, left_on="k", right_index=True)

S = frame_scores()

EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]
CMAP = plt.get_cmap("turbo")
COL = {"found": "limegreen", "missed": "red", "kept": "tab:blue", "not kept": "magenta", "below CUT": "0.6"}

def corners3d(b):
    c2 = bev_corners(b)
    return np.r_[np.c_[c2, np.full(4, b[2] - b[5] / 2)], np.c_[c2, np.full(4, b[2] + b[5] / 2)]]

def camera(k, max_width=1600):
    "The frame's colour image (shrunk to max_width) and the camera matrix for that size; (None, None) if missing."
    if not F["has_image"][k]:
        return None, None
    img = Image.open(str(F["img_path"][k])).convert("RGB")
    cw, ch = F["calib_wh"][k] if F["calib_wh"][k].min() > 0 else img.size   # size the calibration was made for
    if img.size[0] > max_width:
        img = img.resize((max_width, round(img.size[1] * max_width / img.size[0])))
    K = F["K"][k].copy()
    K[0] *= img.size[0] / cw
    K[1] *= img.size[1] / ch
    return img, (K, F["T_cam"][k])

def lidar_points(k, n=30000):
    "The frame's LiDAR points (a random n of them), read from the dataset when the frame is drawn."
    p = str(F["lidar_path"][k])
    pts = in_range(load_points(p, "intensity")) if p else np.zeros((0, 4))
    return pts if len(pts) <= n else pts[np.random.default_rng(0).choice(len(pts), n, replace=False)]

def to_image(xyz, cam):
    "LiDAR-frame points -> pixel (u, v) and depth in front of the camera; cam = (K, LiDAR->camera transform)."
    K, T = cam
    p = np.c_[xyz, np.ones(len(xyz))] @ T.T
    z = p[:, 2]
    uv = (p[:, :3] @ K.T)
    with np.errstate(divide="ignore", invalid="ignore"):
        return uv[:, 0] / z, uv[:, 1] / z, z

def draw_box(ax_cam, ax_bev, cam, b, color, lw, ls="-", label=None):
    x, y = np.r_[bev_corners(b), bev_corners(b)[:1]].T
    ax_bev.plot(x, y, color=color, lw=lw, ls=ls)
    if label:
        ax_bev.text(b[0], b[1] + b[4] / 2 + 0.6, label, color=color, fontsize=11, ha="center", fontweight="bold",
                    clip_on=True)
    if ax_cam is not None and cam is not None:
        u, v, z = to_image(corners3d(b), cam)
        if (z > 0.5).all():
            for a, c in EDGES:
                ax_cam.plot(u[[a, c]], v[[a, c]], color=color, lw=lw, ls=ls)
            if label:
                ax_cam.text(u[4:].mean(), v[4:].min() - 4, label, color=color, fontsize=11, ha="center",
                            fontweight="bold", bbox=dict(fc="white", alpha=.6, lw=0, pad=1), clip_on=True)

def box_names(m, k):
    "A name for every scored predicted box of model m in frame k: a, b, c, ... by score (highest first), then a', b', ..."
    p = PRED[(PRED.model == m) & (PRED.k == k) & (PRED.status != "OUT_OF_RANGE")].sort_values("score", ascending=False)
    return {r.i: chr(97 + n % 26) + "'" * (n // 26) for n, r in enumerate(p.itertuples())}

def owners_by_cutoff(m, k):
    "For each labelled box j of frame k: which predicted box is paired with it at each official score cut-off (-1: none)."
    s = part("pred_" + m, k)
    pb, ps, pc = F["pred_%s_box" % m][s], F["pred_%s_score" % m][s], F["pred_%s_cls" % m][s]
    gc = F["gt_cls"][part("gt", k)]
    inside = np.hypot(pb[:, 0], pb[:, 1]) < MAX_DIST
    own = np.full((len(CUTOFFS), len(gc)), -1)
    for ci, cut in enumerate(CUTOFFS):
        for c in np.unique(gc):
            pi, gj = np.flatnonzero(inside & (pc == c) & (ps >= cut)), np.flatnonzero(gc == c)
            if not len(pi):
                continue
            sub = IOU[m, k][np.ix_(pi, gj)]
            ok = sub >= IOU_THR[CLASSES[c - 1]]
            for r, q in zip(*linear_sum_assignment(np.where(ok, sub, 0.0), maximize=True)):
                if ok[r, q]:
                    own[ci, gj[q]] = pi[r]
    return own

def object_lines(m, k):
    """One line per labelled object (found or missed, by which box, and which box takes it as the score cut-off rises)
    and one line per predicted box (kept = paired with a labelled object at cut-off 0, or not kept and why)."""
    lines, names = [], box_names(m, k)
    g = GT[GT.k == k].sort_values("j")
    p = PRED[(PRED.model == m) & (PRED.k == k) & (PRED.status != "OUT_OF_RANGE")].sort_values("score", ascending=False)
    hit = {r.matched_gt: r for r in p[(p.score >= CUT[m]) & p.status.isin(["TP", "IGNORED"])].itertuples()}
    miss = MISS[(MISS.model == m) & (MISS.k == k)].set_index("j")
    own = owners_by_cutoff(m, k)
    lines.append(("LABELLED OBJECTS", "black"))
    for r in g.itertuples():
        head = "L%d %-7s %2.0fm %3d radar/%4d LiDAR pts" % (r.j + 1, r.cls[:7], r.dist, r.radar_pts, r.lidar_pts)
        if r.j in hit:
            h = hit[r.j]
            txt = "%s  FOUND by box %s (score %.2f, IoU %.2f)" % (head, names[h.i], h.score, IOU[m, k][h.i, r.j])
            color = "green"
        elif r.j in miss.index:
            mm = miss.loc[r.j]
            txt = "%s  MISSED: %s" % (head, mm.reason) + ("" if pd.isna(mm.iou) else ", IoU %.2f" % mm.iou)
            color = "red"
        else:
            txt, color = "%s  (level 2, not scored)" % head, "gray"
        runs, start = [], 0                                     # who holds this object as the cut-off rises
        for ci in range(1, len(CUTOFFS) + 1):
            if ci == len(CUTOFFS) or own[ci, r.j] != own[start, r.j]:
                runs.append("%s %.2f-%.2f" % (names.get(own[start, r.j], "nobody"), CUTOFFS[start], CUTOFFS[ci - 1]))
                start = ci
        if (own[:, r.j] >= 0).any():
            txt += "\n      paired at each cut-off: " + ", ".join(runs)
        lines.append((txt, color))
    lines.append(("PREDICTED BOXES (highest score first)", "black"))
    reason = FALSE[(FALSE.model == m) & (FALSE.k == k)].set_index("i").reason_group
    for r in p.itertuples():
        head = "%-3s %-7s score %.2f, %3d %s pts inside" % (names[r.i], r.cls[:7], r.score, r.pts_inside, SENSOR[m])
        if r.score < CUT[m]:
            lines.append(("%s  below CUT, hidden" % head, COL["below CUT"]))
        elif r.status == "TP":
            lines.append(("%s  KEPT: paired with L%d (IoU %.2f)" % (head, r.matched_gt + 1, IOU[m, k][r.i, r.matched_gt]),
                          COL["kept"]))
        elif r.status == "IGNORED":
            lines.append(("%s  on a level-2 label L%d, not scored" % (head, r.matched_gt + 1), "gray"))
        else:
            holds = [("L%d at cut-offs %.2f-%.2f" % (j + 1, CUTOFFS[np.flatnonzero(own[:, j] == r.i)].min(),
                                                     CUTOFFS[np.flatnonzero(own[:, j] == r.i)].max()))
                     for j in range(own.shape[1]) if (own[:, j] == r.i).any()]
            lines.append(("%s  NOT KEPT (false): %s" % (head, reason.get(r.i, "?"))
                          + ("; but holds " + ", ".join(holds) if holds else ""), COL["not kept"]))
    return lines

LEGEND = [Line2D([], [], color=COL["found"], lw=3, label="labelled object L#, found"),
          Line2D([], [], color=COL["missed"], lw=3, label="labelled object L#, missed"),
          Line2D([], [], color=COL["kept"], lw=2, label="predicted box, KEPT (paired with a label)"),
          Line2D([], [], color=COL["not kept"], lw=2, label="predicted box, NOT KEPT (false)"),
          Line2D([], [], marker="o", ls="", mfc="orange", mec="k", ms=8,
                 label="RADAR point (big dot, colour = distance, see colour bar)"),
          Line2D([], [], marker=".", ls="", color="0.6", label="LiDAR point (small grey dot)")]
RADAR_NORM = plt.Normalize(0, 70)                            # radar dot colour: distance from the car, 0 to 70 m

def draw_points(ax, img, cam, pts, kind):
    "LiDAR (small grey dots) or radar (big dots coloured by distance) on the camera image (cam given) or the top view."
    if not len(pts):
        return
    if kind == "lidar":
        style = dict(s=1, c="0.85" if cam else "0.75", alpha=.5 if cam else 1)
    else:
        style = dict(s=40 if cam else 25, c=CMAP(RADAR_NORM(np.hypot(pts[:, 0], pts[:, 1]))), edgecolors="k",
                     linewidths=.7)
    if cam is None:
        ax.scatter(pts[:, 0], pts[:, 1], **style)
        return
    W, H = img.size
    u, v, z = to_image(pts[:, :3], cam)
    ok = (z > 0.5) & (u >= 0) & (u < W) & (v >= 0) & (v < H)
    if kind != "lidar":
        style["c"] = style["c"][ok]
    ax.scatter(u[ok], v[ok], **style)

def radar_colourbar(fig, ax):
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=RADAR_NORM, cmap=CMAP), ax=ax, orientation="horizontal",
                      fraction=0.05, pad=0.02, aspect=40)
    cb.set_label("radar dot colour = distance from the car (m): blue near, red far", fontsize=11)

def show(frame=None, sample=None, models=("radar_scratch", "radar_ft_from_livox"), both_sensors=False, title="",
         folder="4_frames", inline=True, dpi=None):
    """show(k) with k = row of FR (0..N-1), or show(drive_name, sample_number).
    Top row: the plain camera image, the camera image with only the LiDAR points, and with only the radar points.
    Per model: the camera image with the points of the sensor that model sees (both_sensors=True: both) and its boxes.
    Saves the picture as pictures/<folder>/<weather>_<drive>_s<sample>[_<title>].jpg and shows it (inline=False: save only)."""
    k = int(FR.index[(FR.drive == frame) & (FR["sample"] == sample)][0]) if sample is not None else int(frame)
    models = [m for m in models if m in MODELS]
    gb = F["gt_box"][part("gt", k)]
    rad = F["radar_pts"][part("radar", k)]
    lid = lidar_points(k)
    img, cam = camera(k)
    texts = {m: object_lines(m, k) for m in models}
    n_lines = max(sum(t.count("\n") + 1 for t, _ in texts[m]) for m in models)
    ih = 0.75 if img is None else img.size[1] / img.size[0]
    hr = [ih * len(models) / 3 + 0.12, ih, 0.8, 0.032 * max(n_lines, 2) + 0.05]
    fig = plt.figure(figsize=(13 * len(models), 13 * sum(hr)))
    gs = fig.add_gridspec(4, len(models), height_ratios=hr)
    top = gs[0, :].subgridspec(1, 3, wspace=0.03)
    for i, (what, name) in enumerate([(None, "camera image, nothing drawn on it"),
                                      ("lidar", "only the LiDAR points (%d shown)" % len(lid)),
                                      ("radar", "only the RADAR points (%d)" % len(rad))]):
        ax = fig.add_subplot(top[0, i])
        if img is not None:
            ax.imshow(img)
            if what:
                draw_points(ax, img, cam, lid if what == "lidar" else rad, what)
            ax.set_xlim(0, img.size[0]); ax.set_ylim(img.size[1], 0)
            ax.set_title(name, fontsize=14)
            if what == "radar":
                radar_colourbar(fig, ax)
        else:
            ax.text(.5, .5, "no camera image for this frame", ha="center", transform=ax.transAxes)
        ax.axis("off")
    names_all = {m: box_names(m, k) for m in models}
    kept = [F["pred_%s_box" % m][part("pred_" + m, k)][F["pred_%s_score" % m][part("pred_" + m, k)] >= CUT[m]]
            for m in models]
    xy = np.r_[gb[:, :2].reshape(-1, 2), *[b[:, :2] for b in kept], [[5, -10], [40, 10]]]
    xy = xy[np.hypot(xy[:, 0], xy[:, 1]) < MAX_DIST]
    lim = (max(0, xy[:, 0].min() - 6), min(77, xy[:, 0].max() + 6), xy[:, 1].min() - 6, xy[:, 1].max() + 6)
    for col, m in enumerate(models):
        ax_cam, ax_bev, ax_txt = fig.add_subplot(gs[1, col]), fig.add_subplot(gs[2, col]), fig.add_subplot(gs[3, col])
        if img is not None:
            ax_cam.imshow(img)
            for kind in (["lidar", "radar"] if both_sensors else [SENSOR[m]]):
                draw_points(ax_cam, img, cam, lid if kind == "lidar" else rad, kind)
            ax_cam.set_xlim(0, img.size[0]); ax_cam.set_ylim(img.size[1], 0)
        else:
            ax_cam.text(.5, .5, "no camera image for this frame", ha="center", transform=ax_cam.transAxes)
        ax_cam.axis("off")
        draw_points(ax_bev, None, None, lid, "lidar")             # top view: both sensors
        draw_points(ax_bev, None, None, rad, "radar")
        p = PRED[(PRED.model == m) & (PRED.k == k)]
        pb = F["pred_%s_box" % m][part("pred_" + m, k)]
        names = names_all[m]
        for r in p.sort_values("score").itertuples():           # weakest first, so strong boxes are drawn on top
            if r.status == "OUT_OF_RANGE":
                continue
            if r.score < CUT[m]:
                draw_box(None, ax_bev, cam, pb[r.i], COL["below CUT"], 0.8, "--")
            elif r.status in ("TP", "IGNORED"):
                draw_box(ax_cam, ax_bev, cam, pb[r.i], COL["kept"], 2.2, label="%s=L%d" % (names[r.i], r.matched_gt + 1))
            else:
                draw_box(ax_cam, ax_bev, cam, pb[r.i], COL["not kept"], 1.4, label=names[r.i])
        found = set(p[(p.score >= CUT[m]) & p.status.isin(["TP", "IGNORED"])].matched_gt)
        for j, b in enumerate(gb):
            draw_box(ax_cam, ax_bev, cam, b, COL["found"] if j in found else COL["missed"], 3, label="L%d" % (j + 1))
        ax_bev.set_xlim(lim[0], lim[1]); ax_bev.set_ylim(lim[2], lim[3]); ax_bev.set_aspect("equal")
        ax_bev.set_xlabel("x: metres ahead"); ax_bev.set_ylabel("y: metres to the left"); ax_bev.grid(alpha=.3)
        ax_bev.legend(handles=LEGEND, fontsize=9, loc="upper right", framealpha=.85)
        sc = S[(S.model == m) & (S.k == k)].iloc[0]
        ax_cam.set_title("%s   [camera: %s points]\nfound %d of %d, %d false box(es)  ->  %s    (%s)"
                         % (NAME.get(m, m), "LiDAR + radar" if both_sensors else SENSOR[m].replace("lidar", "LiDAR"),
                            sc.found, sc.labelled, sc.false, sc.verdict,
                            "every box counted" if CUT[m] <= 0 else "score cut-off %.2f" % CUT[m]), fontsize=15)
        ax_txt.axis("off")
        n = 0
        for line, color in texts[m]:
            ax_txt.text(0, 1 - n / max(n_lines, 2), line, color=color, fontsize=10.5, family="monospace", va="top",
                        transform=ax_txt.transAxes, clip_on=True,
                        fontweight="bold" if color == "black" else "normal")
            n += line.count("\n") + 1
    fig.suptitle("%sdrive %s | sample %s | %s / %s | %d radar points in the frame"
                 % (title + "  |  " if title else "", FR.drive[k], FR["sample"][k], FR.weather[k], FR.light[k],
                    len(rad)), fontsize=18)
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    os.makedirs(os.path.join(PIC_DIR, folder), exist_ok=True)
    tag = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")
    path = os.path.join(PIC_DIR, folder, "%s_%s_s%04d%s.jpg" % (FR.weather[k], FR.drive[k], FR["sample"][k],
                                                                 "_" + tag if tag else ""))
    fig.savefig(path, dpi=dpi or SAVE_DPI, pil_kwargs={"quality": 88})
    plt.close(fig)
    if inline:                                   # a smaller copy goes into the notebook, so it stays small to download
        im = Image.open(path)
        im = im.resize((SHOW_WIDTH, round(im.size[1] * SHOW_WIDTH / im.size[0])))
        buf = io.BytesIO(); im.save(buf, "JPEG", quality=80)
        display(ShowImage(data=buf.getvalue(), format="jpeg"))
    return path

show(int(S[S.labelled > 0].k.iloc[len(S[S.labelled > 0]) // 2]))

## 5. GOOD, BAD and UGLY frames, per weather

For one model (`MODEL`), the frames it handles best and worst in each weather, each drawn next to the other radar
model so you can see what differs. Picks are spread over different drives.

In [ ]:
MODEL = "radar_ft_from_livox"          # whose verdict picks the frames
COMPARE = ("radar_scratch", "radar_ft_from_livox")
N_EACH = 2                              # frames per verdict and weather
SHOW_WEATHERS = WEATHERS                # or e.g. ["Fog", "Heavy_Rain"]

def pick(model, verdict, weather, n):
    d = S[(S.model == model) & (S.weather == weather) & (S.labelled > 0) & FR.has_image.reindex(S.k).values]
    if verdict == "GOOD":
        d = d[d.verdict == "GOOD"].sort_values("labelled", ascending=False)
    elif verdict == "BAD":
        d = d[d.missed > 0].sort_values(["missed", "found"], ascending=[False, True])
    else:
        d = d[d.false > 0].sort_values("false", ascending=False)
    first = d.drop_duplicates("drive")                     # one per drive first, then the rest
    return list(pd.concat([first, d.drop(first.index)]).k.head(n))

for w in SHOW_WEATHERS:
    for verdict in ["GOOD", "BAD", "UGLY"]:
        for k in pick(MODEL, verdict, w, N_EACH):
            show(k, models=COMPARE, title="%s for %s" % (verdict, NAME[MODEL]),
                 folder="5_good_bad_ugly")

## 6. Where the LiDAR pre-training made the difference

Frames where the fine-tuned radar model finds the most labelled objects that radar-from-scratch misses (and the
other way round), per weather. The object list under each picture says exactly what each model did.

In [ ]:
A, B = "radar_scratch", "radar_ft_from_livox"
N_EACH = 2
d = S[S.model == A][["k", "found", "false"]].merge(S[S.model == B][["k", "found", "false"]], on="k", suffixes=("_A", "_B"))
d["gain"] = d.found_B - d.found_A
d = d.merge(FR, left_on="k", right_index=True)
for w in WEATHERS:
    x = d[(d.weather == w) & d.has_image]
    for label, sel in [("fine-tuned finds more", x[x.gain > 0].sort_values("gain", ascending=False)),
                       ("scratch finds more", x[x.gain < 0].sort_values("gain"))]:
        first = sel.drop_duplicates("drive")
        for k in pd.concat([first, sel.drop(first.index)]).k.head(N_EACH):
            show(k, models=(A, B), title="%s: %s" % (w, label), folder="6_pretraining_difference")

## 6b. Standing vs moving vehicles

The labels carry no speed, so each labelled vehicle's speed is measured from the radar's own Doppler value (first
lines of the cell). "Standing" covers parked cars and cars waiting in traffic. The speed is the part along the line of
sight, so a car crossing exactly sideways also counts as standing. Vehicles without a radar point have no speed and
form their own group.

1. Vehicle AP per speed group. The groups use the speed limits of Waymo's velocity breakdown (0.2 / 1 / 3 / 10 m/s)
   and are scored the way Waymo scores that breakdown; the matching and the AP are those of section 2.
2. The same, standing vs moving, per weather.
3. Plain AP (exactly as section 2) over the frames that hold only standing, both, or only moving vehicles.
4. Per weather, the frames with the most standing vehicles missed. Their pictures go to `pictures/6b_standing_missed/`.

This split is not in the DSERT-RoLL paper. It is a diagnosis next to the paper's numbers.

In [ ]:
# ---- 1) how fast every labelled vehicle moves, measured by the radar itself ----------------------------------
# The 4th value of a radar point is its raw Doppler speed along the line of sight. For anything that stands still it
# equals (speed of our own car) x cos(angle to the point). So per frame: fit our own speed from all points (most of
# the scene stands still), subtract it, and what is left is the point's own speed over the ground along the line of
# sight. A vehicle's speed = the median of that over the radar points inside its labelled box.
SPEED_EDGES = [0.2, 1.0, 3.0, 10.0]   # m/s, the limits of Waymo's velocity breakdown (STATIONARY/SLOW/MEDIUM/FAST/VERY_FAST)
SPEED_NAMES = ["standing (< 0.2 m/s)", "0.2 - 1 m/s", "1 - 3 m/s", "3 - 10 m/s", "> 10 m/s", "speed unknown (no radar point)"]
MIN_POINTS_FOR_EGO_FIT = 20           # frames with fewer radar points: speed unknown

own_speed = np.full(N, np.nan)        # our own car, m/s
gt_speed = np.full(len(F["gt_box"]), np.nan)
left_over = []
for k in range(N):
    p = F["radar_pts"][part("radar", k)].astype(float)
    if len(p) < MIN_POINTS_FOR_EGO_FIT:
        continue
    cos = p[:, 0] / np.maximum(np.hypot(p[:, 0], p[:, 1]), 1e-6)
    own_speed[k] = np.median(p[cos > 0.5, 3] / cos[cos > 0.5])
    rest = p[:, 3] - own_speed[k] * cos
    left_over.append(rest)
    for j, b in enumerate(F["gt_box"][part("gt", k)].astype(float)):
        d = p[:, :3] - b[:3]
        c, s = np.cos(-b[6]), np.sin(-b[6])
        inside = ((np.abs(d[:, 0] * c - d[:, 1] * s) <= b[3] / 2) & (np.abs(d[:, 0] * s + d[:, 1] * c) <= b[4] / 2)
                  & (np.abs(d[:, 2]) <= b[5] / 2))
        if inside.any():
            gt_speed[F["gt_off"][k] + j] = abs(np.median(rest[inside]))
left_over = np.abs(np.concatenate(left_over))
print("check of the method: %.1f%% of all radar points are within 0.2 m/s of 'standing still' after subtracting our own "
      "speed (%.1f%% within 0.5 m/s)" % (100 * (left_over < 0.2).mean(), 100 * (left_over < 0.5).mean()))
print("our own speed: median %.0f km/h, standing (< 0.5 m/s) in %d of %d frames"
      % (3.6 * np.nanmedian(np.abs(own_speed)), (np.abs(own_speed) < 0.5).sum(), N))

gt_group = np.where(np.isnan(gt_speed), 5, np.digitize(gt_speed, SPEED_EDGES))    # 0..4 = the speed bins, 5 = unknown
GT["speed"] = gt_speed[F["gt_off"][GT.k.values] + GT.j.values]
GT["speed_group"] = gt_group[F["gt_off"][GT.k.values] + GT.j.values]
SETS = {n: [i] for i, n in enumerate(SPEED_NAMES)}
SETS["moving (0.2 m/s or faster)"] = [1, 2, 3, 4]
ORDER = [SPEED_NAMES[0], "moving (0.2 m/s or faster)"] + SPEED_NAMES[1:]

# ---- 2) AP per speed group, the way Waymo scores a breakdown that only the labels have (its velocity breakdown,
# detection_metrics.cc): the matching is the normal one, between all predicted and all labelled boxes, redone at
# every score cut-off. For one group: a hit counts if the labelled box it is paired with is in the group; a
# labelled box of the group without a partner is a miss; an unpaired predicted box is a false box for the group
# unless the labelled box it overlaps most (IoU >= 0.01) is in another group (then it is left out of this group).
def tp_fp_by_group(m, cls, group, sets):
    """TP, FP: (frames, cut-offs) over all labelled boxes, as tp_fp_per_cutoff.
    TPs, FPs: (frames, cut-offs, sets), one per set of group numbers; group = one number per labelled box (cache order)."""
    c = CLASSES.index(cls) + 1
    member = np.zeros((len(sets), int(group.max()) + 1), bool)
    for i, g in enumerate(sets):
        member[i, g] = True
    TP = np.zeros((N, len(CUTOFFS)), np.int32); FP = np.zeros_like(TP)
    TPs = np.zeros((N, len(CUTOFFS), len(sets)), np.int32); FPs = np.zeros_like(TPs)
    for k in range(N):
        s = part("pred_" + m, k)
        pb, ps, pc = F["pred_%s_box" % m][s], F["pred_%s_score" % m][s], F["pred_%s_cls" % m][s]
        pi = np.flatnonzero((pc == c) & (np.hypot(pb[:, 0], pb[:, 1]) < MAX_DIST))
        if not len(pi):
            continue
        gc, gl = F["gt_cls"][part("gt", k)], F["gt_level"][part("gt", k)]
        gj = np.flatnonzero(gc == c)
        gg = group[F["gt_off"][k] + gj]
        above = ps[pi][None, :] >= CUTOFFS[:, None]
        counts = np.ones((len(sets), len(pi)), bool)          # would this box be a false box for the set, if unpaired?
        tp = np.zeros(len(CUTOFFS), np.int32); ign = np.zeros_like(tp)
        tps = np.zeros((len(CUTOFFS), len(sets)), np.int32); paired = np.zeros_like(tps)
        if len(gj):
            sub = IOU[m, k][np.ix_(pi, gj)]
            near = gg[sub.argmax(1)]                           # group of the labelled box it overlaps most
            counts = np.where(sub.max(1) >= 0.01, member[:, near], True)
            ok = sub >= IOU_THR[cls]
            cand = np.flatnonzero(ok.any(1))
            if len(cand):
                cand = cand[np.argsort(-ps[pi][cand], kind="stable")]
                n_cand = above[:, cand].sum(1)
                for n in np.unique(n_cand):
                    r_, q_ = linear_sum_assignment(np.where(ok[cand[:n]], sub[cand[:n]], 0.0), maximize=True)
                    hit = [(cand[r], q) for r, q in zip(r_, q_) if ok[cand[r], q]]
                    at = n_cand == n
                    tp[at] = sum(gl[gj[q]] == 1 for _, q in hit)
                    ign[at] = sum(gl[gj[q]] != 1 for _, q in hit)
                    if hit:
                        tps[at] = sum(member[:, gg[q]] * (gl[gj[q]] == 1) for _, q in hit)
                        paired[at] = sum(counts[:, r] for r, _ in hit)
        TP[k], FP[k] = tp, above.sum(1) - tp - ign
        TPs[k], FPs[k] = tps, above.astype(np.int32) @ counts.T.astype(np.int32) - paired
    return TP, FP, TPs, FPs

SPEED_MODELS = [m for m in ["radar_scratch", "radar_ft_from_livox", "livox_pretrain"] if m in MODELS]
VEH = {m: tp_fp_by_group(m, "Vehicle", gt_group, list(SETS.values())) for m in SPEED_MODELS}
V = GT[(GT.cls == "Vehicle") & (GT.level == 1)]
n_veh = V.groupby("k").size().reindex(range(N), fill_value=0).values                       # labelled vehicles per frame
n_set = {n: V[V.speed_group.isin(g)].groupby("k").size().reindex(range(N), fill_value=0).values for n, g in SETS.items()}

def speed_table(mask):
    rows = {}
    for i, n in enumerate(SETS):
        row = {"labelled vehicles": int(n_set[n][mask].sum())}
        for m in SPEED_MODELS:
            ap, rec, _ = ap_of(VEH[m][2][mask][:, :, i], VEH[m][3][mask][:, :, i], n_set[n][mask].sum())
            row["AP " + NAME.get(m, m)], row["found " + NAME.get(m, m)] = round(ap, 4), "%.1f%%" % (100 * rec)
        rows[n] = row
    return pd.DataFrame(rows).T.loc[ORDER]

for m in SPEED_MODELS:                 # nothing split -> must be the numbers of section 2
    assert abs(ap_of(VEH[m][0], VEH[m][1], n_veh.sum())[0] - AP[m, "Vehicle", "all"]) < 1e-6
print("\nVehicle AP by how fast the vehicle moves (all frames). 'found' = share of the labelled vehicles with a hit, every box counted.")
display(speed_table(np.ones(N, bool)))

rows = {}
for w in WEATHERS:
    t = speed_table((FR.weather == w).values)
    a, b = t.loc[SPEED_NAMES[0]], t.loc["moving (0.2 m/s or faster)"]
    row = {"standing": a["labelled vehicles"], "moving": b["labelled vehicles"],
           "share standing": "%.0f%%" % (100 * a["labelled vehicles"] / max(a["labelled vehicles"] + b["labelled vehicles"], 1))}
    for m in SPEED_MODELS:
        row["AP standing | " + NAME.get(m, m)], row["AP moving | " + NAME.get(m, m)] = a["AP " + NAME.get(m, m)], b["AP " + NAME.get(m, m)]
    rows[w] = row
print("\nPer weather: standing vs moving vehicles (vehicles with a radar point on them)")
display(pd.DataFrame(rows).T)

# ---- 3) the same split by frame: plain AP (as in section 2) over three sets of frames -------------------------
known = V[V.speed_group < 5]
n_stand = known[known.speed_group == 0].groupby("k").size().reindex(range(N), fill_value=0).values
n_move = known[known.speed_group > 0].groupby("k").size().reindex(range(N), fill_value=0).values
FRAME_SETS = {"frames with only standing vehicles": (n_stand > 0) & (n_move == 0),
              "frames with standing and moving vehicles": (n_stand > 0) & (n_move > 0),
              "frames with only moving vehicles": (n_stand == 0) & (n_move > 0)}
rows = {}
for n, mask in FRAME_SETS.items():
    row = {"frames": int(mask.sum()), "labelled vehicles": int(n_veh[mask].sum())}
    for m in SPEED_MODELS:
        row["AP " + NAME.get(m, m)] = round(ap_of(VEH[m][0][mask], VEH[m][1][mask], n_veh[mask].sum())[0], 4)
    rows[n] = row
print("\nVehicle AP over frames sorted by what is in them (every labelled vehicle of the frame counts, also those without a radar point)")
display(pd.DataFrame(rows).T)

# ---- 4) the frames to look at: most standing vehicles missed, per weather ---------------------------------------
MISSED_BY_MODEL = "radar_ft_from_livox"
N_EACH = 2                              # pictures per weather (0 = only the list)
N_LISTED = 8                            # frames listed per weather
for m in SPEED_MODELS:
    p = PRED[(PRED.model == m) & PRED.status.isin(["TP", "IGNORED"])]
    V = V.assign(**{"found_" + m: pd.MultiIndex.from_arrays([V.k, V.j]).isin(pd.MultiIndex.from_arrays([p.k, p.matched_gt]))})
st = V[V.speed_group == 0]
agg = {"standing vehicles": ("j", "size")}
agg.update({"missed by " + NAME.get(m, m): ("found_" + m, lambda v: int((~v).sum())) for m in SPEED_MODELS})
PARKED = st.groupby("k").agg(**agg).join(FR[["weather", "drive", "sample", "light", "has_image"]])
PARKED["picture name starts with"] = PARKED.weather + "_" + PARKED.drive + "_s" + PARKED["sample"].map("{:04d}".format)
key = "missed by " + NAME.get(MISSED_BY_MODEL, MISSED_BY_MODEL)
for w in WEATHERS:
    d = PARKED[(PARKED.weather == w) & (PARKED[key] > 0)].sort_values([key, "standing vehicles"], ascending=False)
    first = d.drop_duplicates("drive")                     # one per drive first, then the rest
    d = pd.concat([first, d.drop(first.index)])
    print("\n%s: %d frames with a standing vehicle that '%s' missed (first column = frame number k)"
          % (w, len(d), NAME.get(MISSED_BY_MODEL, MISSED_BY_MODEL)))
    display(d.drop(columns=["weather", "has_image"]).head(N_LISTED))
    for k in d[d.has_image].index[:N_EACH]:
        show(int(k), models=SPEED_MODELS, title="standing vehicles missed", folder="6b_standing_missed")

## 6c. Worst and best drives

Every validation drive with its Vehicle AP (the drive's frames only, computed as in section 2), sorted worst first for
the model in `RANK_BY`. Then `N_FRAMES` pictures of each of the `N_DRIVES` worst and best drives, saved in one folder
per drive: `pictures/6c_worst_and_best_drives/worst_1_<weather>_<drive>/`, `.../best_1_<weather>_<drive>/`, and so on.

In [ ]:
RANK_BY = "radar_ft_from_livox"        # the model whose Vehicle AP ranks the drives
N_DRIVES = 3                            # this many worst and this many best drives get pictures
N_FRAMES = 4                            # pictures per drive, spread evenly over the drive's frames with a labelled vehicle
MIN_LABELLED = 50                       # drives with fewer labelled vehicles are listed but get no pictures
DRIVE_MODELS = SPEED_MODELS             # models in the table and in the pictures

rows = {}
for d, x in FR.groupby("drive"):
    mask = (FR.drive == d).values
    v = V[V.drive == d]
    row = {"weather": x.weather.iloc[0], "light": x.light.iloc[0], "frames": len(x), "labelled vehicles": len(v),
           "standing": "%.0f%%" % (100 * (v.speed_group == 0).sum() / max((v.speed_group < 5).sum(), 1)),
           "no radar point": "%.0f%%" % (100 * (v.radar_pts == 0).mean()) if len(v) else "",
           "median distance (m)": round(v.dist.median()) if len(v) else np.nan}
    for m in DRIVE_MODELS:
        ap, rec, _ = ap_of(VEH[m][0][mask], VEH[m][1][mask], n_veh[mask].sum())
        row["AP " + NAME.get(m, m)] = round(ap, 3) if len(v) else np.nan
        row["found " + NAME.get(m, m)] = "%.0f%%" % (100 * rec) if len(v) else ""
    rows[d] = row
DRIVES = pd.DataFrame(rows).T.sort_values("AP " + NAME.get(RANK_BY, RANK_BY))
print("Every validation drive, worst Vehicle AP of '%s' first (AP over the drive's frames, computed as in section 2)"
      % NAME.get(RANK_BY, RANK_BY))
with pd.option_context("display.max_rows", 200):
    display(DRIVES)

ranked = DRIVES[DRIVES["labelled vehicles"] >= MIN_LABELLED]
picks = [("worst", i + 1, d) for i, d in enumerate(ranked.index[:N_DRIVES])] + \
        [("best", i + 1, d) for i, d in enumerate(ranked.index[::-1][:N_DRIVES])]
for which, rank, d in picks:
    r = DRIVES.loc[d]
    print("\n%s %d: drive %s | %s / %s | %d labelled vehicles, %s standing | %s"
          % (which.upper(), rank, d, r.weather, r.light, r["labelled vehicles"], r.standing,
             ", ".join("%s AP %.3f" % (NAME.get(m, m), r["AP " + NAME.get(m, m)]) for m in DRIVE_MODELS)))
    ks = FR.index[(FR.drive == d).values & FR.has_image.values & (n_veh > 0)]
    for k in ([ks[int(i)] for i in np.linspace(0, len(ks) - 1, N_FRAMES)] if len(ks) > N_FRAMES else ks):
        show(int(k), models=DRIVE_MODELS, title="%s drive %d" % (which, rank),
             folder="6c_worst_and_best_drives/%s_%d_%s_%s" % (which, rank, r.weather, d))

## 6d. Frames by number of vehicles

How the models do on frames with 0, 1, 2, 3, 4 or more labelled vehicles (and "more than 2", "more than 3"):
the GOOD / BAD / UGLY verdicts of section 5 and the Vehicle AP over those frames.

In [ ]:
# Frames sorted by how many labelled vehicles they hold. Verdicts are those of section 5 (all classes, every box
# counted): GOOD = nothing missed and no false box, BAD = something missed, UGLY = a false box, BAD+UGLY = both.
COUNT_SETS = {"0 vehicles": n_veh == 0, "1 vehicle": n_veh == 1, "2 vehicles": n_veh == 2, "3 vehicles": n_veh == 3,
              "4 or more": n_veh >= 4, "more than 2": n_veh > 2, "more than 3": n_veh > 3, "all frames": n_veh >= 0}
for m in SPEED_MODELS:
    s = S[S.model == m].set_index("k").verdict.reindex(range(N))
    false_veh = PRED[(PRED.model == m) & (PRED.cls == "Vehicle") & (PRED.status == "FP")].groupby("k").size() \
        .reindex(range(N), fill_value=0).values
    rows = {}
    for n, mask in COUNT_SETS.items():
        v = s[mask].value_counts()
        ap, rec, _ = ap_of(VEH[m][0][mask], VEH[m][1][mask], n_veh[mask].sum())
        rows[n] = {"frames": int(mask.sum()), **{c: int(v.get(c, 0)) for c in ["GOOD", "BAD", "UGLY", "BAD+UGLY"]},
                   "labelled vehicles": int(n_veh[mask].sum()),
                   "found": "%.1f%%" % (100 * rec) if n_veh[mask].sum() else "",
                   "false vehicle boxes per frame": round(false_veh[mask].mean(), 1),
                   "Vehicle AP": round(ap, 4) if n_veh[mask].sum() else ""}
    print("\n%s" % NAME.get(m, m))
    display(pd.DataFrame(rows).T)

## 6e. Right from above, wrong in height

For each labelled vehicle, is there a box that fits it seen from above (top-view IoU >= 0.5), and does it still fail
the 3D test because it sits too high or too low? Run this for every new model: the row "lost between the two" is the
part of the misses that comes from height alone.

In [ ]:
# For every labelled vehicle: the model's vehicle box that fits it best seen from above (largest top-view IoU), and how
# far that box is off vertically. This is a diagnosis next to the AP of section 2, not a replacement for it.
def top_view_iou(a, b):
    if np.hypot(a[0] - b[0], a[1] - b[1]) > (np.hypot(a[3], a[4]) + np.hypot(b[3], b[4])) / 2:
        return 0.0
    inter = area(clip_polygon(bev_corners(a), bev_corners(b)))
    return inter / max(a[3] * a[4] + b[3] * b[4] - inter, 1e-9)

rows = []
for m in SPEED_MODELS:
    for k in np.flatnonzero(n_veh > 0):
        gb, gc = F["gt_box"][part("gt", k)].astype(float), F["gt_cls"][part("gt", k)]
        s = part("pred_" + m, k)
        pb, pc = F["pred_%s_box" % m][s].astype(float), F["pred_%s_cls" % m][s]
        pi = np.flatnonzero((pc == 1) & (np.hypot(pb[:, 0], pb[:, 1]) < MAX_DIST))
        for j in np.flatnonzero(gc == 1):
            top = np.array([top_view_iou(pb[i], gb[j]) for i in pi])
            if not len(top) or top.max() <= 0:
                rows.append((m, k, j, 0.0, 0.0, np.nan, np.nan, gb[j, 2], np.hypot(gb[j, 0], gb[j, 1])))
                continue
            i = pi[top.argmax()]
            rows.append((m, k, j, IOU[m, k][pi, j].max(), top.max(), pb[i, 2] - gb[j, 2], pb[i, 5] - gb[j, 5], gb[j, 2],
                         np.hypot(gb[j, 0], gb[j, 1])))
H = pd.DataFrame(rows, columns=["model", "k", "j", "iou3d", "iou_top", "dz", "dh", "true_z", "dist"])

rows = {}
for m in SPEED_MODELS:
    h = H[H.model == m]
    ok = h[h.iou_top >= 0.5]                                  # the box is right seen from above
    rows[NAME.get(m, m)] = {
        "labelled vehicles": len(h),
        "a box with 3D IoU >= 0.5 exists": "%.1f%%" % (100 * (h.iou3d >= 0.5).mean()),
        "a box with top-view IoU >= 0.5 exists": "%.1f%%" % (100 * (h.iou_top >= 0.5).mean()),
        "lost between the two (points)": round(100 * ((h.iou_top >= 0.5).mean() - (h.iou3d >= 0.5).mean()), 1),
        "z error of that box, median |m|": round(ok.dz.abs().median(), 2),
        "z error within 0.2 m": "%.0f%%" % (100 * (ok.dz.abs() < 0.2).mean()),
        "box height error, median |m|": round(ok.dh.abs().median(), 2),
        "predicted z follows true z (correlation)": round(np.corrcoef(ok.dz + ok.true_z, ok.true_z)[0, 1], 2),
        "z error if every box sat at the average height, median |m|": round((ok.true_z - ok.true_z.median()).abs().median(), 2)}
print("Vehicles: is the box right from above but wrong in height?")
display(pd.DataFrame(rows))

t = H.assign(distance=pd.cut(H.dist, [0, 20, 40, MAX_DIST], labels=["0-20 m", "20-40 m", "40-70 m"]))
g = t.groupby(["distance", "model"], observed=False)
lost = 100 * (g.iou_top.apply(lambda v: (v >= 0.5).mean()) - g.iou3d.apply(lambda v: (v >= 0.5).mean()))
print("\nPoints of 'found' lost between top view and 3D, by distance:")
display(lost.unstack().rename(columns=NAME).round(1))

## 7. Look at anything you want

Set `WEATHER`, `DRIVE`, `SAMPLES` (or `FRAMES`), optionally `MISSED_BY` / `NO_RADAR_BOX`, and run the cell: it draws the
matching frames (at most `N_MAX`) for the models in `MODELS_SHOWN`.

**Every picture in this notebook is also saved** as a `.jpg` in the output folder under `pictures/` (one sub-folder per
section: `4_frames`, `5_good_bad_ugly`, `6_pretraining_difference`, `6b_standing_missed`,
`6c_worst_and_best_drives/<worst or best>_<n>_<weather>_<drive>`, `7_my_picks`, `drive_<name>`), and this cell ends by
packing them all into **`pictures.zip`**. On Kaggle: Output tab (or the folder icon on the right) -> `pictures.zip` ->
download. `save_drive(drive)` saves one picture per frame of a whole drive without showing them.

In [ ]:
def save_drive(drive, every=1, models=("radar_scratch", "radar_ft_from_livox")):
    "One picture per frame (every n-th) of a drive into pictures/drive_<drive>/, saved only, small resolution."
    ks = list(FR.index[FR.drive == drive])[::every]
    for k in ks:
        show(k, models=models, folder="drive_" + drive, inline=False, dpi=50)
    print("%d pictures in %s" % (len(ks), os.path.join(PIC_DIR, "drive_" + drive)))

# ---- Look at anything: set what you want, run the cell. None = any. ----
WEATHER = None        # "Clear", "Light_Rain", "Heavy_Rain", "Fog", "Light_Snow", "Heavy_Snow", or None
DRIVE = None          # a drive name, e.g. "2025_04_16_07_04_30", or None
SAMPLES = None        # one sample number, a list [10, 20, 30], range(0, 200, 20), or None
FRAMES = None         # frame numbers k as printed elsewhere in this notebook (e.g. [111, 112]); overrides the three above
MISSED_BY = None      # only frames where this model missed a labelled object, e.g. "radar_ft_from_livox", or None
NO_RADAR_BOX = False  # True: only frames with a labelled object that has no radar point on it
MODELS_SHOWN = ("radar_scratch", "radar_ft_from_livox")   # add "livox_pretrain" / "livox_e20_on_radar" to compare
N_MAX = 4             # draw at most this many of the matching frames (spread over the matches)
FOLDER = "7_my_picks" # sub-folder of pictures/ the drawings are saved in

if FRAMES is not None:
    ks = [int(k) for k in np.atleast_1d(FRAMES)]
else:
    sel = FR.copy()
    if WEATHER:
        sel = sel[sel.weather == WEATHER]
    if DRIVE:
        sel = sel[sel.drive == DRIVE]
    if SAMPLES is not None:
        sel = sel[sel["sample"].isin([int(s) for s in np.atleast_1d(list(SAMPLES) if isinstance(SAMPLES, range) else SAMPLES)])]
    if MISSED_BY:
        sel = sel[sel.index.isin(S[(S.model == MISSED_BY) & (S.missed > 0)].k)]
    if NO_RADAR_BOX:
        sel = sel[sel.index.isin(GT[(GT.level == 1) & (GT.radar_pts == 0)].k)]
    ks = list(sel.index)
    if not DRIVE:
        print("drives matching:", sel.groupby(["weather", "drive"]).size().to_dict())
print("%d frames match" % len(ks), "" if len(ks) <= N_MAX else "-> drawing %d of them, spread over the matches" % N_MAX)
for k in ([ks[int(i)] for i in np.linspace(0, len(ks) - 1, N_MAX)] if len(ks) > N_MAX else ks):
    show(k, models=MODELS_SHOWN, folder=FOLDER)

# A whole drive, one picture per frame, saved only (not shown here), to flip through like a video:
# save_drive("2025_04_16_07_04_30", every=2)

zip_pictures()   # every picture saved so far -> pictures.zip in the output folder (Kaggle: Output tab -> download)